# 4\.2\.4 Establish Forecasting Baselines and Forecastability

Notebook 4.2.3 completed the leakage-safe forecasting handoff: each native modeling track now has a historical forecasting matrix at **Modeling Track × Forecast Sequence × Origin × Horizon** grain, with the future observed mobility value retained strictly as the prediction target.

This notebook establishes the forecasting floor that every more complex model must later beat.

The goal is deliberately modest. We are not yet fitting ARIMA/SARIMAX, VARIMA, tree-based, or neural models. Instead, we ask how well a small set of simple historical rules can forecast mobility when they are subjected to the same rolling-origin experiment that later models will face.

These baselines serve two related purposes:

1. **Establish a benchmark.** A more complex forecasting model is useful only if it can improve meaningfully on a strong, transparent historical rule.
2. **Assess empirical forecastability.** If recent or recurring historical patterns already predict future mobility reasonably well out of sample, the series contains exploitable forecasting structure. If simple rules perform poorly or deteriorate sharply with horizon, that tells us the forecasting problem is intrinsically harder — not that a target should automatically be abandoned.

"Forecastability" is therefore used here in an empirical rather than formal-test sense. We are not conducting a single statistical test that labels a series forecastable or unforecastable. We will instead examine how much error simple historical structure produces, whether recurring patterns improve on basic persistence, how broadly those improvements occur across Forecast Sequences, and how performance changes from h=1 to h=2 to h=5.

## Evaluation principles

Baseline selection is based on **rolling-validation MAE only**. RMSE, normalized MAE, bias, coverage, and improvement relative to persistence will be reported to provide a fuller picture, but they do not choose the winning benchmark.

The final holdout remains untouched during benchmark selection.

The **Post-CP Primary** experiment defines the benchmark carried into the later Post-CP model comparison. The **Full-History Regime-Aware** track does not receive a separately optimized benchmark: its purpose is to test whether adding pre-CP historical information improves forecasting of the same post-CP era. Post-CP Primary and Full-History will therefore ultimately be compared against the same frozen benchmark on their common supported population.

The **Pre-CP Reference** is a different forecasting problem and receives its own benchmark, selected only from its legal pre-CP rolling-validation period.

Native modeling-track populations remain intact throughout. Shared populations are used only where a direct cross-track comparison requires them.

## Reader-facing visual strategy

The main results will emphasize comparisons that can be understood without knowing forecasting terminology. Planned views include:

- a **"How much better than persistence?" heatmap** showing percentage MAE improvement by target and horizon;
- a compact **baseline winner matrix** identifying the strongest simple rule for each Target × Horizon;
- a **horizon-decay view** showing how forecast error changes as the target moves farther from the forecast origin;
- a **share-of-series view** showing how often the winning benchmark actually improves on persistence across individual Forecast Sequences rather than only in the aggregate;
- selected observed-versus-baseline examples where a concrete time series helps explain what the aggregate scores mean.

The objective is for readers to see not just which baseline wins, but what "winning" means in practical forecasting terms.

In [1]:
# ---------------------------------------------------------------------
# 4.2.4.0A — Setup and fixed evaluation contract
# ---------------------------------------------------------------------

from pathlib import Path
import gc
import math

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display


# ---------------------------------------------------------------------
# 1) Project paths
# ---------------------------------------------------------------------
# Keep 4.2.4 physically separate from the completed 4.2.3 handoff.
# Intermediate prediction surfaces may be large, so they get their own
# cache directory rather than being mixed with final downstream assets.

PROJECT_ROOT = Path(".")
PIPELINE_DIR = PROJECT_ROOT / "pipeline_data"

INPUT_411_DIR = PIPELINE_DIR / "4.1.1.final_tables"
INPUT_412_DIR = PIPELINE_DIR / "4.1.2.final_tables"
INPUT_423_DIR = PIPELINE_DIR / "4.2.3.final_tables"

INTERMEDIATE_DIR = PIPELINE_DIR / "4.2.4.intermediate_tables"
FINAL_DIR = PIPELINE_DIR / "4.2.4.final_tables"

INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# 2) Authoritative upstream inputs
# ---------------------------------------------------------------------
# The 4.2.3 historical matrices define the rows we are allowed to score.
# The earlier sequence and rolling-origin registries remain authoritative
# for canonical time ordering and experiment timing.

FORECAST_SEQUENCE_PATH = INPUT_411_DIR / "forecasting_target_sequence_panel.parquet"
ROLLING_ORIGIN_PATH = INPUT_412_DIR / "rolling_origin_registry.parquet"
TRACK_CONTRACT_PATH = INPUT_412_DIR / "forecasting_track_contract_registry.parquet"

FINAL_QA_423_PATH = INPUT_423_DIR / "final_forecasting_matrix_qa.parquet"
FINAL_MANIFEST_423_PATH = INPUT_423_DIR / "forecasting_matrix_manifest.parquet"

FINAL_MATRIX_PATHS = {
    "post_cp_primary": INPUT_423_DIR / "forecasting_matrix__post_cp_primary.parquet",
    "full_history_regime_aware": INPUT_423_DIR / "forecasting_matrix__full_history_regime_aware.parquet",
    "pre_cp_reference": INPUT_423_DIR / "forecasting_matrix__pre_cp_reference.parquet",
}


# ---------------------------------------------------------------------
# 3) Planned 4.2.4 outputs
# ---------------------------------------------------------------------
# Candidate-level baseline prediction surfaces remain detailed evaluation
# assets in intermediate_tables. The selected Pre-CP benchmark surface is
# published separately in final_tables for downstream Chapter 5 use.

BASELINE_PREDICTION_PATHS = {
    track: INTERMEDIATE_DIR / f"baseline_predictions__{track}.parquet"
    for track in FINAL_MATRIX_PATHS
}

BASELINE_DEFINITION_PATH = FINAL_DIR / "baseline_definition_registry.parquet"
BASELINE_LEADERBOARD_PATH = FINAL_DIR / "baseline_leaderboard.parquet"
FROZEN_BENCHMARK_PATH = FINAL_DIR / "advanced_model_benchmark_registry.parquet"
SELECTED_PRE_CP_BENCHMARK_PREDICTIONS_PATH = (
    FINAL_DIR / "selected_benchmark_predictions__pre_cp_reference.parquet"
)
FORECASTABILITY_SUMMARY_PATH = FINAL_DIR / "empirical_forecastability_summary.parquet"
FINAL_QA_424_PATH = FINAL_DIR / "final_baseline_forecastability_qa.parquet"
FINAL_MANIFEST_424_PATH = FINAL_DIR / "baseline_forecastability_manifest.parquet"


# ---------------------------------------------------------------------
# 4) Forecasting contract inherited from 4.2.1–4.2.3
# ---------------------------------------------------------------------

FORECAST_TARGETS = [
    "taxi_trip_count", "taxi_avg_trip_speed", "fhvhv_trip_count",
    "fhvhv_avg_trip_speed", "subway_ridership",
]

FORECAST_HORIZONS = [1, 2, 5]

MODELING_TRACKS = [
    "post_cp_primary",
    "full_history_regime_aware",
    "pre_cp_reference",
]

TRACK_LABELS = {
    "post_cp_primary": "Post-CP Primary",
    "full_history_regime_aware": "Full-History Regime-Aware",
    "pre_cp_reference": "Pre-CP Reference",
}

TARGET_LABELS = {
    "taxi_trip_count": "Taxi trips",
    "taxi_avg_trip_speed": "Taxi average speed",
    "fhvhv_trip_count": "FHVHV trips",
    "fhvhv_avg_trip_speed": "FHVHV average speed",
    "subway_ridership": "Subway ridership",
}

DAYPART_ORDER = ["overnight", "am_peak", "midday", "pm_peak", "evening"]

DAYPART_LABELS = {
    "overnight": "Overnight",
    "am_peak": "AM peak",
    "midday": "Midday",
    "pm_peak": "PM peak",
    "evening": "Evening",
}

TEMPORAL_BUCKET_LABELS = {
    "weekday_overnight": "Weekday · Overnight",
    "weekday_am_peak": "Weekday · AM peak",
    "weekday_midday": "Weekday · Midday",
    "weekday_pm_peak": "Weekday · PM peak",
    "weekday_evening": "Weekday · Evening",
    "weekend_overnight": "Weekend · Overnight",
    "weekend_am_peak": "Weekend · AM peak",
    "weekend_midday": "Weekend · Midday",
    "weekend_pm_peak": "Weekend · PM peak",
    "weekend_evening": "Weekend · Evening",
}

STEPS_PER_DAY = 5
STEPS_PER_WEEK = 35

EXPECTED_NATIVE_SEQUENCE_COUNTS = {
    "post_cp_primary": 1_108,
    "full_history_regime_aware": 1_051,
    "pre_cp_reference": 1_039,
}

# This cohort is used only when Post-CP and Full-History are compared directly.
# Native populations remain intact for their ordinary track-level summaries.
POST_FULL_COMMON_SEQUENCE_COUNT = 1_042


# ---------------------------------------------------------------------
# 5) Baseline-evaluation rules already locked
# ---------------------------------------------------------------------

SELECTION_METRIC = "mae"

# Reuse the 4.2.1 comparable-history support convention rather than
# inventing a new missing-history threshold specifically for baselines.
ROLLING_MIN_OBS_SHARE = 0.80
TRAILING_HISTORY_DAYS = 7
TRAILING_MIN_OBS = math.ceil(TRAILING_HISTORY_DAYS * ROLLING_MIN_OBS_SHARE)

WRITE_INTERMEDIATES = True
WRITE_FINAL_OUTPUTS = True
RUN_VISUALS = True


# ---------------------------------------------------------------------
# 6) Small shared helpers
# ---------------------------------------------------------------------

def qa_row(check, passed, detail):
    """Return one compact PASS/FAIL QA record."""
    return {"check": check, "status": "PASS" if bool(passed) else "FAIL", "detail": detail}


def parquet_columns(path):
    """Inspect column names without materializing a large Parquet file."""
    return pq.ParquetFile(path).schema_arrow.names


def parquet_metadata(path):
    """Read lightweight file metadata without loading the forecasting matrix."""
    pf = pq.ParquetFile(path)
    return {
        "rows": int(pf.metadata.num_rows),
        "columns": int(pf.metadata.num_columns),
        "row_groups": int(pf.metadata.num_row_groups),
        "size_gb": path.stat().st_size / (1024 ** 3),
    }


# ---------------------------------------------------------------------
# 7) Fail early if the completed forecasting handoff is unavailable
# ---------------------------------------------------------------------
# Baseline construction must consume the closed 4.2.3 package. It should
# never silently rebuild the forecasting experiment from raw mobility data.

required_inputs = [
    FORECAST_SEQUENCE_PATH, ROLLING_ORIGIN_PATH, TRACK_CONTRACT_PATH,
    FINAL_QA_423_PATH, FINAL_MANIFEST_423_PATH, *FINAL_MATRIX_PATHS.values(),
]

missing_inputs = [str(path) for path in required_inputs if not path.exists()]

if missing_inputs:
    raise FileNotFoundError(
        "4.2.4 is missing required upstream inputs:\n" + "\n".join(missing_inputs)
    )

print(
    f"4.2.4 setup complete · {len(MODELING_TRACKS)} modeling tracks · "
    f"{len(FORECAST_TARGETS)} targets · horizons {FORECAST_HORIZONS}"
)

4.2.4 setup complete · 3 modeling tracks · 5 targets · horizons [1, 2, 5]


### Precommit the baseline evaluation rules

Before looking at any baseline error, fix the rules that determine what counts as a winner.

This is important because seemingly minor choices — selecting on RMSE after seeing MAE, choosing a different benchmark for Full-History after observing its results, or letting holdout performance influence the preferred baseline — would make the later model comparison easier to optimize after the fact.

The benchmark tournament therefore has one primary selection criterion: **lowest rolling-validation MAE**.

Other error summaries remain valuable, but they answer different questions. RMSE emphasizes occasional large misses, normalized MAE makes targets with different numerical scales easier to compare, and bias shows whether a rule systematically forecasts too high or too low. We will also translate MAE into an intuitive **percentage improvement versus persistence** when possible.

None of those secondary views overrides the validation-MAE winner.

In [2]:
# ---------------------------------------------------------------------
# 4.2.4.0B — Reader-facing baseline evaluation contract
# ---------------------------------------------------------------------

baseline_evaluation_contract_df = pd.DataFrame([
    {"Decision": "Winner-selection metric", "Locked rule": "Lowest rolling-validation MAE"},
    {"Decision": "Additional reported metrics", "Locked rule": "RMSE, normalized MAE, bias, coverage, and MAE improvement vs persistence"},
    {"Decision": "Holdout use", "Locked rule": "Never used to choose or revise a benchmark"},
    {"Decision": "Post-CP benchmark", "Locked rule": "Selected from Post-CP Primary rolling validation"},
    {"Decision": "Full-History benchmark", "Locked rule": "Reuse the Post-CP Primary benchmark; do not optimize a separate one"},
    {"Decision": "Pre-CP benchmark", "Locked rule": "Select separately using only Pre-CP Reference rolling validation"},
    {"Decision": "Track populations", "Locked rule": "Preserve native populations; restrict only for explicit cross-track comparisons"},
    {"Decision": "Forecastability interpretation", "Locked rule": "Empirical out-of-sample evidence; no arbitrary pass/fail threshold"},
])

display(baseline_evaluation_contract_df)

,Decision,Locked rule
0,Winner-selection metric,Lowest rolling-validation MAE
1,Additional reported metrics,"RMSE, normalized MAE, bias, coverage, and MAE ..."
2,Holdout use,Never used to choose or revise a benchmark
3,Post-CP benchmark,Selected from Post-CP Primary rolling validation
4,Full-History benchmark,Reuse the Post-CP Primary benchmark; do not op...
5,Pre-CP benchmark,Select separately using only Pre-CP Reference ...
6,Track populations,Preserve native populations; restrict only for...
7,Forecastability interpretation,Empirical out-of-sample evidence; no arbitrary...


**Findings.** The evaluation rules are fixed before any baseline performance is observed. Rolling-validation MAE alone selects the benchmark, the final holdout cannot influence that selection, and Full-History must compete against the same benchmark chosen for the Post-CP Primary forecasting problem. This keeps the later comparison focused on the substantive question — whether richer modeling or additional pre-CP history improves forecasting — rather than allowing each experiment to choose the easiest reference point.

---

## 4.2.4.1 Validate the Baseline-Evaluation Handoff

Baseline construction should begin from the completed 4.2.3 production artifacts rather than reconstructing the forecasting experiment.

This section therefore performs a lightweight handoff audit. It does not rerun 4.2.3's expensive feature engineering or repeat every leakage check already closed upstream. Instead, it verifies that:

- the three production matrices still match the 4.2.3 manifest;
- the completed 4.2.3 QA remains fully passing;
- each modeling track still exposes its native Forecast Sequence population;
- h=1, h=2, and h=5 remain correctly aligned to their target positions;
- rolling-validation periods match the upstream track contract;
- the Post-CP and Full-History matrices retain their shared comparison population;
- future labels remain explicitly separated from the forecast information available at the origin.

Only the evaluation stages needed by this notebook are materialized for the second audit. Initial-training rows do not need to be loaded simply to confirm the baseline evaluation population, because the historical donor values used by the baseline rules will be resolved separately from the authoritative target-sequence history.

In [3]:
# ---------------------------------------------------------------------
# 4.2.4.1A — Validate the completed 4.2.3 production handoff
# ---------------------------------------------------------------------

# ---------------------------------------------------------------------
# 1) Confirm that the upstream notebook closed cleanly
# ---------------------------------------------------------------------

final_qa_423_df = pd.read_parquet(FINAL_QA_423_PATH)
manifest_423_df = pd.read_parquet(FINAL_MANIFEST_423_PATH)

required_qa_columns = {"status"}
required_manifest_columns = {"path", "rows", "columns"}

missing_qa_columns = sorted(
    required_qa_columns - set(final_qa_423_df.columns)
)

missing_manifest_columns = sorted(
    required_manifest_columns - set(manifest_423_df.columns)
)

if missing_qa_columns:
    raise ValueError(
        f"4.2.3 final QA is missing required fields: {missing_qa_columns}"
    )

if missing_manifest_columns:
    raise ValueError(
        f"4.2.3 manifest is missing required fields: {missing_manifest_columns}"
    )

upstream_pass_count = int(final_qa_423_df["status"].eq("PASS").sum())
upstream_qa_complete = upstream_pass_count == len(final_qa_423_df)


# ---------------------------------------------------------------------
# 2) Build a filename lookup from the final 4.2.3 manifest
# ---------------------------------------------------------------------
# Compare only metadata here. Loading the multi-million-row matrices would
# be unnecessary just to prove that the final production files still match
# the handoff that 4.2.3 recorded.

manifest_423_df = manifest_423_df.copy()
manifest_423_df["filename"] = (
    manifest_423_df["path"].astype(str).map(lambda value: Path(value).name)
)

duplicate_manifest_filenames = (
    manifest_423_df.loc[
        manifest_423_df["filename"].duplicated(keep=False),
        "filename",
    ]
    .dropna()
    .unique()
    .tolist()
)

if duplicate_manifest_filenames:
    raise ValueError(
        "4.2.3 manifest contains duplicate filenames: "
        f"{sorted(duplicate_manifest_filenames)}"
    )

manifest_lookup = manifest_423_df.set_index("filename")

expected_matrix_filenames = {
    path.name for path in FINAL_MATRIX_PATHS.values()
}

missing_manifest_entries = sorted(
    expected_matrix_filenames - set(manifest_lookup.index)
)

if missing_manifest_entries:
    raise ValueError(
        "4.2.3 manifest is missing forecasting-matrix entries: "
        f"{missing_manifest_entries}"
    )


# ---------------------------------------------------------------------
# 3) Compare each final forecasting matrix with its recorded manifest entry
# ---------------------------------------------------------------------

inventory_records = []

for track in MODELING_TRACKS:
    path = FINAL_MATRIX_PATHS[track]
    metadata = parquet_metadata(path)

    manifest_row = manifest_lookup.loc[path.name]
    manifest_rows = int(manifest_row["rows"])
    manifest_columns = int(manifest_row["columns"])

    manifest_match = (
        metadata["rows"] == manifest_rows
        and metadata["columns"] == manifest_columns
    )

    inventory_records.append({
        "modeling_track": TRACK_LABELS[track],
        "rows": metadata["rows"],
        "columns": metadata["columns"],
        "row_groups": metadata["row_groups"],
        "size_gb": round(metadata["size_gb"], 3),
        "manifest_rows": manifest_rows,
        "manifest_columns": manifest_columns,
        "manifest_match": manifest_match,
        "upstream_4_2_3_qa": f"{upstream_pass_count}/{len(final_qa_423_df)} PASS",
    })

matrix_handoff_inventory_df = pd.DataFrame(inventory_records)
display(matrix_handoff_inventory_df)


# ---------------------------------------------------------------------
# 4) Stop immediately if the supposedly closed upstream handoff changed
# ---------------------------------------------------------------------

if not upstream_qa_complete:
    raise AssertionError("4.2.4 cannot proceed because one or more completed 4.2.3 QA checks failed.")

if not matrix_handoff_inventory_df["manifest_match"].all():
    raise AssertionError("One or more historical forecasting matrices no longer match the 4.2.3 manifest.")

,modeling_track,rows,columns,row_groups,size_gb,manifest_rows,manifest_columns,manifest_match,upstream_4_2_3_qa
0,Post-CP Primary,7486756,120,9,1.029,7486756,120,True,74/74 PASS
1,Full-History Regime-Aware,18688882,120,21,2.477,18688882,120,True,74/74 PASS
2,Pre-CP Reference,11384323,120,13,1.485,11384323,120,True,74/74 PASS


**Findings.** The completed 4.2.3 forecasting package carries forward exactly as expected. The Post-CP Primary, Full-History Regime-Aware, and Pre-CP Reference matrices contain 7,486,756, 18,688,882, and 11,384,323 rows respectively, with 120 columns in each file, and every file matches the row and column counts recorded in the 4.2.3 manifest. The full upstream audit also remains at 74/74 checks passing. We can therefore begin baseline evaluation from the finalized forecasting matrices without rebuilding or reconciling any upstream data.

### Validate the rolling evaluation populations

The next check looks only at the rows that can affect baseline evaluation.

For the two post-CP forecasting tracks, this means rolling validation plus final holdout. For the Pre-CP Reference, ordinary historical evaluation stops at rolling validation; its January 2025 refit-only period serves a later application role rather than acting as another model-selection test set.

Reading only these stages keeps the handoff audit lightweight while still checking the populations on which baseline scores and later model scores must be comparable.

In [4]:
# ---------------------------------------------------------------------
# 4.2.4.1B — Validate evaluation-stage populations and horizon alignment
# ---------------------------------------------------------------------

# ---------------------------------------------------------------------
# 1) Limit the audit to fields required to prove evaluation comparability
# ---------------------------------------------------------------------
# We deliberately avoid loading the remaining ~100 predictor/provenance
# columns from the large 4.2.3 matrices.

MATRIX_EVALUATION_AUDIT_COLUMNS = [
    "modeling_track", "stage", "metric", "forecast_sequence_id",
    "origin_observation_sequence_id", "target_observation_sequence_id",
    "horizon", "target_date", "target_observed_value",
    "target_label_available", "native_track_eligible",
    "post_cp_common_comparison_eligible",
]

track_contract_df = pd.read_parquet(
    TRACK_CONTRACT_PATH,
    columns=[
        "modeling_track",
        "rolling_validation_target_start_date",
        "rolling_validation_target_end_date",
    ],
)

for column in ["rolling_validation_target_start_date", "rolling_validation_target_end_date"]:
    track_contract_df[column] = pd.to_datetime(track_contract_df[column]).dt.normalize()

# ---------------------------------------------------------------------
# 2) Require exactly one contract row for each modeling track
# ---------------------------------------------------------------------
# A duplicate or missing track would make the validation-window contract
# ambiguous, so fail explicitly rather than silently choosing one row.

required_track_contract_df = track_contract_df.loc[
    track_contract_df["modeling_track"].isin(MODELING_TRACKS)
].copy()

contract_counts = (
    required_track_contract_df["modeling_track"]
    .value_counts()
    .reindex(MODELING_TRACKS, fill_value=0)
)

invalid_contract_counts = contract_counts.loc[contract_counts.ne(1)]

if not invalid_contract_counts.empty:
    raise ValueError(
        "Expected exactly one forecasting contract row per modeling track; "
        f"observed counts = {invalid_contract_counts.to_dict()}"
    )

track_contract_lookup = required_track_contract_df.set_index("modeling_track")

# ---------------------------------------------------------------------
# 3) Audit each modeling track independently
# ---------------------------------------------------------------------
# Native track populations must survive untouched. The common Post/Full
# cohort is checked as an additional comparison flag, not used to filter
# the ordinary track-level evaluation surface.

evaluation_summary_records = []
evaluation_failures = []

for track in MODELING_TRACKS:
    path = FINAL_MATRIX_PATHS[track]

    missing_columns = sorted(set(MATRIX_EVALUATION_AUDIT_COLUMNS) - set(parquet_columns(path)))
    if missing_columns:
        raise ValueError(f"{track} is missing required 4.2.4 audit columns: {missing_columns}")

    # Pre-CP Reference has no ordinary final holdout. The other two tracks do.
    stage_filter = (
        [("stage", "==", "rolling_validation")]
        if track == "pre_cp_reference"
        else [("stage", "in", ["rolling_validation", "final_holdout"])]
    )

    audit = pd.read_parquet(path, columns=MATRIX_EVALUATION_AUDIT_COLUMNS, filters=stage_filter)
    audit["target_date"] = pd.to_datetime(audit["target_date"]).dt.normalize()

    # Repeated string identifiers are categorical purely to reduce memory use.
    for column in ["modeling_track", "stage", "metric", "forecast_sequence_id"]:
        audit[column] = audit[column].astype("category")

    track_values = set(audit["modeling_track"].dropna().unique().tolist())
    metric_values = set(audit["metric"].dropna().unique().tolist())
    stages = sorted(str(value) for value in audit["stage"].dropna().unique().tolist())
    horizons = sorted(int(value) for value in audit["horizon"].dropna().unique().tolist())

    forecast_sequence_count = int(audit["forecast_sequence_id"].nunique())
    duplicate_keys = int(
        audit.duplicated(
            ["forecast_sequence_id", "origin_observation_sequence_id", "horizon"]
        ).sum()
    )

    # Every target must sit exactly h canonical positions after its origin.
    horizon_alignment_ok = (
        audit["target_observation_sequence_id"]
        .sub(audit["origin_observation_sequence_id"])
        .eq(audit["horizon"])
        .all()
    )

    # The explicit label-availability flag must agree with the actual outcome.
    label_flag_ok = audit["target_label_available"].eq(audit["target_observed_value"].notna()).all()

    # Every row in a native production matrix should remain native-track eligible.
    native_flag_ok = audit["native_track_eligible"].fillna(False).astype(bool).all()

    validation = audit.loc[audit["stage"].eq("rolling_validation")]
    observed_validation_start = validation["target_date"].min() if not validation.empty else pd.NaT
    observed_validation_end = validation["target_date"].max() if not validation.empty else pd.NaT

    contract_row = track_contract_lookup.loc[track]
    expected_validation_start = contract_row["rolling_validation_target_start_date"]
    expected_validation_end = contract_row["rolling_validation_target_end_date"]

    validation_window_ok = (
        observed_validation_start == expected_validation_start
        and observed_validation_end == expected_validation_end
    )

    # Only Post-CP and Full-History participate in the direct shared-cohort test.
    if track in {"post_cp_primary", "full_history_regime_aware"}:
        common_mask = audit["post_cp_common_comparison_eligible"].fillna(False).astype(bool)
        common_sequence_count = int(audit.loc[common_mask, "forecast_sequence_id"].nunique())

        holdout_stage_ok = "final_holdout" in stages
        common_population_ok = common_sequence_count == POST_FULL_COMMON_SEQUENCE_COUNT
    else:
        common_sequence_count = np.nan
        holdout_stage_ok = True
        common_population_ok = True

    checks = {
        "track_identity": track_values == {track},
        "target_set": metric_values == set(FORECAST_TARGETS),
        "horizons": horizons == FORECAST_HORIZONS,
        "native_sequences": forecast_sequence_count == EXPECTED_NATIVE_SEQUENCE_COUNTS[track],
        "duplicate_keys": duplicate_keys == 0,
        "target_alignment": horizon_alignment_ok,
        "label_flag": label_flag_ok,
        "native_flag": native_flag_ok,
        "validation_window": validation_window_ok,
        "holdout_stage": holdout_stage_ok,
        "post_full_common_population": common_population_ok,
    }

    failed_checks = [name for name, passed in checks.items() if not passed]
    if failed_checks:
        evaluation_failures.append(f"{track}: {failed_checks}")

    evaluation_summary_records.append({
        "modeling_track": TRACK_LABELS[track],
        "evaluation_rows": len(audit),
        "forecast_sequences": forecast_sequence_count,
        "horizons": ", ".join(map(str, horizons)),
        "evaluation_stages": ", ".join(stages),
        "validation_start": observed_validation_start,
        "validation_end": observed_validation_end,
        "target_label_coverage_pct": round(audit["target_label_available"].mean() * 100, 3),
        "post_full_common_sequences": common_sequence_count,
        "status": "PASS" if not failed_checks else "FAIL",
    })

    # Release each multi-million-row audit surface before loading the next track.
    del audit, validation
    gc.collect()


# ---------------------------------------------------------------------
# 3) Present one compact reader-facing handoff summary
# ---------------------------------------------------------------------

evaluation_handoff_summary_df = pd.DataFrame(evaluation_summary_records)
display(evaluation_handoff_summary_df)


# ---------------------------------------------------------------------
# 4) Fail before baseline construction if evaluation populations drifted
# ---------------------------------------------------------------------

if evaluation_failures:
    raise AssertionError(
        "4.2.4 evaluation handoff failed:\n" + "\n".join(evaluation_failures)
    )

,modeling_track,evaluation_rows,forecast_sequences,horizons,evaluation_stages,validation_start,validation_end,target_label_coverage_pct,post_full_common_sequences,status
0,Post-CP Primary,5999820,1108,"1, 2, 5","final_holdout, rolling_validation",2025-04-05,2026-01-04,99.509,1042.0,PASS
1,Full-History Regime-Aware,5691165,1051,"1, 2, 5","final_holdout, rolling_validation",2025-04-05,2026-01-04,99.527,1042.0,PASS
2,Pre-CP Reference,5704110,1039,"1, 2, 5",rolling_validation,2024-01-01,2024-12-31,99.588,NaN,PASS


**Findings.** All three evaluation populations pass the handoff checks. Post-CP Primary retains 1,108 Forecast Sequences, Full-History Regime-Aware retains 1,051, and Pre-CP Reference retains 1,039, with all three supporting h=1, h=2, and h=5. The Post-CP and Full-History tracks also preserve the same 1,042-series comparison population needed for their later head-to-head test. Their rolling-validation period runs from April 5, 2025 through January 4, 2026, while the Pre-CP Reference correctly uses calendar year 2024 for validation and has no ordinary post-CP holdout. Observed target values are available for roughly 99.5% of evaluation rows, and every structural check passes, so the baseline scores can be compared on the intended forecasting experiment without changing the populations or time splits.

### Section Summary

The baseline experiment now inherits the completed forecasting architecture without redefining it.

The production matrices remain the source of the eligible forecast rows, while the rolling-validation boundaries and horizon arithmetic remain those established by the upstream forecasting contract. Post-CP Primary and Full-History retain their native populations together with their explicit shared comparison population; the Pre-CP Reference remains a separate pre-policy forecasting experiment.

No baseline has been evaluated yet, and no holdout performance has been inspected for model-selection purposes.

---

## 4.2.4.2 Construct Simple Forecasting Baselines

A baseline forecast is a rule for choosing an earlier observed value — or a summary of earlier observed values — and using it as the prediction for a specific future target.

That means its historical donor must be aligned to the **target being forecast**, not merely to the forecast origin.

For example, suppose the forecast origin is Monday AM Peak:

- h=1 predicts Monday Midday;
- h=2 predicts Monday PM Peak;
- h=5 predicts Tuesday AM Peak.

A "same Daypart yesterday" forecast for h=1 must therefore use **Sunday Midday**. Using an origin-side feature describing Sunday AM Peak would answer a different historical question.

This distinction matters because the 4.2.1 historical predictors were correctly constructed to describe conditions known **at the origin**. Baseline forecasting now asks a different question: *which legal historical observation is the best simple stand-in for this particular target period?*

The baseline rules will therefore resolve their own target-aligned historical donors from the authoritative ordered Forecast Sequence history while enforcing a hard rule that every donor must occur at or before the forecast origin.

### Baseline rules already fixed

**Previous-step persistence** carries the value observed at the forecast origin forward to the target. It asks: *what if mobility simply stays where it is now?*

**Previous-day same Daypart** uses the target Daypart exactly one calendar day earlier. With five canonical Dayparts per day, its donor is exactly `target_observation_sequence_id - 5`.

**Shifted trailing historical mean** averages the seven most recent observations of the target Daypart that are legally available by the forecast origin. It requires at least six usable values, matching the established 80% comparable-history support convention. Missing historical mobility is not filled from future observations merely to make the baseline available.

At h=5, persistence and previous-day same-Daypart necessarily use the same donor: h=5 predicts the same Daypart on the following day, so the forecast origin itself is exactly one day before the target. That expected tie is a property of the horizon definition rather than a data error.

### One definition still requires human review

The project plan also calls for a **previous comparable Daypart** baseline. There are two defensible leakage-safe meanings, and choosing between them changes the historical comparison being tested.

Before generating millions of baseline predictions, inspect concrete target examples and lock that definition.

In [5]:
# ---------------------------------------------------------------------
# 4.2.4.2A — Illustrate target-aligned baseline donor timing
# ---------------------------------------------------------------------
#
# This cell uses timing only. It does NOT score a baseline and does NOT
# inspect the target outcome we are eventually trying to predict.
#
# Candidate interpretations of "previous comparable Daypart":
#
#   Option A — same Daypart exactly one week earlier
#              fixed seasonal-naive lag = 35 canonical positions
#
#   Option B — latest legally available period with the same target
#              temporal bucket = weekday/weekend × Daypart
#
# We use Post-CP rolling-validation examples only so every illustrated
# donor sits comfortably inside the legal historical information window.
# ---------------------------------------------------------------------


# ---------------------------------------------------------------------
# 1) Recover one copy of the universal canonical Date × Daypart calendar
# ---------------------------------------------------------------------
# Every Forecast Sequence shares the same 5,930 canonical positions.
# We therefore need only one sequence to illustrate donor timing.

post_pf = pq.ParquetFile(FINAL_MATRIX_PATHS["post_cp_primary"])
sample_sequence_table = post_pf.read_row_group(0, columns=["forecast_sequence_id"])
sample_forecast_sequence_id = sample_sequence_table.column("forecast_sequence_id")[0].as_py()

canonical_calendar_df = pd.read_parquet(
    FORECAST_SEQUENCE_PATH,
    columns=[
        "forecast_sequence_id", "observation_sequence_id",
        "date", "daypart", "temporal_bucket",
    ],
    filters=[("forecast_sequence_id", "==", sample_forecast_sequence_id)],
)

canonical_calendar_df["date"] = pd.to_datetime(canonical_calendar_df["date"]).dt.normalize()
canonical_calendar_df = canonical_calendar_df.sort_values("observation_sequence_id").reset_index(drop=True)

if canonical_calendar_df["observation_sequence_id"].duplicated().any():
    raise ValueError("Sample Forecast Sequence does not contain a unique canonical calendar.")


# ---------------------------------------------------------------------
# 2) Attach readable origin and target timing to the rolling schedule
# ---------------------------------------------------------------------
# The rolling-origin registry supplies the experiment schedule. Calendar
# labels come from the canonical sequence so the illustration uses exactly
# the same ordering as every downstream forecasting matrix.

rolling_schedule_df = pd.read_parquet(
    ROLLING_ORIGIN_PATH,
    columns=[
        "modeling_track", "stage", "observation_sequence_id",
        "target_observation_sequence_id", "horizon",
    ],
).rename(columns={"observation_sequence_id": "origin_observation_sequence_id"})

origin_calendar_lookup = canonical_calendar_df[
    ["observation_sequence_id", "date", "daypart", "temporal_bucket"]
].rename(columns={
    "observation_sequence_id": "origin_observation_sequence_id",
    "date": "origin_date",
    "daypart": "origin_daypart",
    "temporal_bucket": "origin_temporal_bucket",
})

target_calendar_lookup = canonical_calendar_df[
    ["observation_sequence_id", "date", "daypart", "temporal_bucket"]
].rename(columns={
    "observation_sequence_id": "target_observation_sequence_id",
    "date": "target_date",
    "daypart": "target_daypart",
    "temporal_bucket": "target_temporal_bucket",
})

illustration_schedule_df = (
    rolling_schedule_df
    .merge(
        origin_calendar_lookup,
        on="origin_observation_sequence_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        target_calendar_lookup,
        on="target_observation_sequence_id",
        how="left",
        validate="many_to_one",
    )
)

illustration_schedule_df["target_date"] = pd.to_datetime(
    illustration_schedule_df["target_date"]
).dt.normalize()


# ---------------------------------------------------------------------
# 3) Choose one weekday and one weekend AM target
# ---------------------------------------------------------------------
# These examples make the distinction intuitive:
# - weekly seasonal-naive always looks exactly seven days backward;
# - same temporal bucket may use a much more recent weekday/weekend peer.

example_candidates = illustration_schedule_df.loc[
    illustration_schedule_df["modeling_track"].eq("post_cp_primary")
    & illustration_schedule_df["stage"].eq("rolling_validation")
    & illustration_schedule_df["horizon"].eq(1)
    & illustration_schedule_df["target_daypart"].eq("am_peak")
].copy()

example_candidates["target_weekday"] = example_candidates["target_date"].dt.dayofweek


def choose_weekday_example(day_of_week):
    """Choose the earliest rolling-validation AM target for one weekday."""
    candidates = (
        example_candidates.loc[example_candidates["target_weekday"].eq(day_of_week)]
        .sort_values(["target_date", "target_observation_sequence_id"])
    )

    if candidates.empty:
        raise ValueError(f"No donor-illustration target found for weekday {day_of_week}.")

    return candidates.iloc[0]


# Monday exposes weekday recurrence; Saturday exposes weekend recurrence.
example_rows = [choose_weekday_example(0), choose_weekday_example(5)]
calendar_by_id = canonical_calendar_df.set_index("observation_sequence_id")


# ---------------------------------------------------------------------
# 4) Define tiny presentation helpers
# ---------------------------------------------------------------------

def format_period(row):
    """Return one reader-friendly Date × Daypart label."""
    date_label = pd.Timestamp(row["date"]).strftime("%a %b %d, %Y")
    daypart_label = DAYPART_LABELS[row["daypart"]]
    return f"{date_label} · {daypart_label}"


def latest_same_bucket_donor(target_temporal_bucket, origin_observation_sequence_id):
    """
    Return the latest canonical period at or before the forecast origin
    sharing the target's weekday/weekend × Daypart temporal bucket.
    """
    candidates = canonical_calendar_df.loc[
        canonical_calendar_df["temporal_bucket"].eq(target_temporal_bucket)
        & canonical_calendar_df["observation_sequence_id"].le(origin_observation_sequence_id)
    ]

    return None if candidates.empty else candidates.iloc[-1]


# ---------------------------------------------------------------------
# 5) Resolve the competing donor definitions for concrete examples
# ---------------------------------------------------------------------
# These are timing donors only. Historical target values are intentionally
# not joined yet because this cell exists to support a methodological HITL
# decision, not to peek at which option forecasts better.

donor_illustration_records = []

for example in example_rows:
    origin_id = int(example["origin_observation_sequence_id"])
    target_id = int(example["target_observation_sequence_id"])

    origin_period = calendar_by_id.loc[origin_id]
    target_period = calendar_by_id.loc[target_id]

    # Fixed calendar offsets from the TARGET, not from the origin.
    previous_day_period = calendar_by_id.loc[target_id - STEPS_PER_DAY]
    previous_week_period = calendar_by_id.loc[target_id - STEPS_PER_WEEK]

    # Option B varies with weekday/weekend structure and current origin timing.
    same_bucket_period = latest_same_bucket_donor(
        example["target_temporal_bucket"],
        origin_id,
    )

    # Show which same-Daypart periods would form the 7-day trailing candidate set.
    trailing_periods = canonical_calendar_df.loc[
        canonical_calendar_df["daypart"].eq(example["target_daypart"])
        & canonical_calendar_df["observation_sequence_id"].le(origin_id)
    ].tail(TRAILING_HISTORY_DAYS)

    trailing_span = (
        f"{format_period(trailing_periods.iloc[0])} → {format_period(trailing_periods.iloc[-1])}"
        if len(trailing_periods) == TRAILING_HISTORY_DAYS
        else f"{len(trailing_periods)} prior candidate periods"
    )

    donor_illustration_records.append({
        "example": f"{pd.Timestamp(example['target_date']).day_name()} AM target",
        "target_bucket": TEMPORAL_BUCKET_LABELS[
            example["target_temporal_bucket"]
        ],
        "forecast_origin": format_period(origin_period),
        "forecast_target": format_period(target_period),
        "persistence_donor": format_period(origin_period),
        "previous_day_same_daypart": format_period(previous_day_period),
        "option_a_previous_week_same_daypart": format_period(previous_week_period),
        "option_b_latest_same_temporal_bucket": (
            format_period(same_bucket_period)
            if same_bucket_period is not None
            else "Unavailable"
        ),
        "trailing_7d_same_daypart_span": trailing_span,
    })


# ---------------------------------------------------------------------
# 6) Present the timing examples for human review
# ---------------------------------------------------------------------

baseline_donor_illustration_df = pd.DataFrame(donor_illustration_records)
display(baseline_donor_illustration_df)

,example,target_bucket,forecast_origin,forecast_target,persistence_donor,previous_day_same_daypart,option_a_previous_week_same_daypart,option_b_latest_same_temporal_bucket,trailing_7d_same_daypart_span
0,Monday AM target,Weekday · AM peak,"Mon Apr 07, 2025 · Overnight","Mon Apr 07, 2025 · AM peak","Mon Apr 07, 2025 · Overnight","Sun Apr 06, 2025 · AM peak","Mon Mar 31, 2025 · AM peak","Fri Apr 04, 2025 · AM peak","Mon Mar 31, 2025 · AM peak → Sun Apr 06, 2025 ..."
1,Saturday AM target,Weekend · AM peak,"Sat Apr 05, 2025 · Overnight","Sat Apr 05, 2025 · AM peak","Sat Apr 05, 2025 · Overnight","Fri Apr 04, 2025 · AM peak","Sat Mar 29, 2025 · AM peak","Sun Mar 30, 2025 · AM peak","Sat Mar 29, 2025 · AM peak → Fri Apr 04, 2025 ..."


**Findings.** The examples confirm that baseline history must be chosen relative to the future period being predicted, not simply relative to the forecast origin. For a Monday AM target, persistence uses Monday Overnight, the previous-day rule uses Sunday AM, and the previous-week rule uses the prior Monday AM. The alternative "latest same temporal bucket" rule instead reaches back to Friday AM because Sunday is classified as weekend. For a Saturday AM target, the same rule reaches back to the previous Sunday AM. This makes the fixed previous-week definition easier to understand and more consistent across the calendar: persistence tests immediate continuity, the previous-day baseline tests daily repetition, and the previous-week baseline cleanly tests weekly repetition.

> ✅ Decision — Use Previous Week, Same Daypart
What simple historical patterns should our baseline tournament test?

We will define the previous comparable Daypart candidate as the target period exactly 7 days earlier \(target\_observation\_sequence\_id \- 35\)\. This gives us four simple, distinct questions to ask before introducing fitted forecasting models:

\* Persistence: What if the current mobility level simply continues?
\* Previous day, same Daypart: What if this Daypart looks like the same Daypart yesterday?
\* Previous week, same Daypart: What if this Daypart looks like the same Daypart last week?
\* Recent 7\-day same\-Daypart mean: What if this Daypart looks like its recent one\-week average?

The fixed previous\-week definition keeps those ideas meaningfully different: persistence tests immediate continuity, the previous\-day rule tests daily repetition, the previous\-week rule tests weekly repetition, and the trailing mean tests a smoothed recent pattern\. It is also easier to explain and avoids the inconsistent calendar jumps produced by the alternative weekday/weekend\-bucket definition\.

These four rules are candidate baselines, not four downstream benchmarks, and none is a fitted statistical model\. They compete to establish the simple forecasting floor that more complex models will later need to beat\.

Their results ultimately feed two benchmark contracts:

\* the Post\-CP forecasting benchmark, selected from Post\-CP Primary validation and then reused unchanged for the Post\-CP Primary versus Full\-History Regime\-Aware experiment; and
\* the Pre\-CP counterfactual benchmark, selected separately from the legal Pre\-CP Reference validation period and later carried into the no\-congestion\-pricing counterfactual work\.

In [6]:
# ---------------------------------------------------------------------
# 4.2.4.2B — Register the four candidate baseline rules
# ---------------------------------------------------------------------

# ---------------------------------------------------------------------
# 1) Define stable machine IDs and reader-facing descriptions
# ---------------------------------------------------------------------
# These are candidate forecasting RULES, not four downstream benchmark
# contracts. Their stable IDs will later identify which rule wins each
# Target × Horizon slot in the Post-CP forecasting and Pre-CP benchmark registries.

BASELINE_SPECS = [
    {
        "baseline_id": "persistence",
        "baseline_label": "Persistence",
        "plain_english_rule": "Carry the mobility value at the forecast origin forward.",
        "historical_donor_rule": "origin_observation_sequence_id",
        "structure_tested": "Immediate level continuity",
    },
    {
        "baseline_id": "previous_day_same_daypart",
        "baseline_label": "Previous day · same Daypart",
        "plain_english_rule": "Use the target Daypart from exactly one calendar day earlier.",
        "historical_donor_rule": "target_observation_sequence_id - 5",
        "structure_tested": "Daily repetition",
    },
    {
        "baseline_id": "previous_week_same_daypart",
        "baseline_label": "Previous week · same Daypart",
        "plain_english_rule": "Use the target Daypart from exactly seven calendar days earlier.",
        "historical_donor_rule": "target_observation_sequence_id - 35",
        "structure_tested": "Weekly repetition",
    },
    {
        "baseline_id": "trailing_7d_same_daypart_mean",
        "baseline_label": "Recent 7-day Daypart mean",
        "plain_english_rule": (
            "Average the previous seven observations of the target Daypart, "
            "requiring at least six observed values."
        ),
        "historical_donor_rule": (
            "target Daypart at -5, -10, -15, -20, -25, -30, and -35 positions"
        ),
        "structure_tested": "Smoothed recent Daypart level",
    },
]

BASELINE_COLUMNS = {
    spec["baseline_id"]: f"baseline__{spec['baseline_id']}"
    for spec in BASELINE_SPECS
}


# ---------------------------------------------------------------------
# 2) Create the compact candidate-rule registry
# ---------------------------------------------------------------------

baseline_definition_registry_df = pd.DataFrame(BASELINE_SPECS).assign(
    fitted_model=False,
    target_aligned=True,
    selection_candidate=True,
)

if baseline_definition_registry_df["baseline_id"].duplicated().any():
    raise ValueError("Candidate baseline registry contains duplicate baseline IDs.")

if set(BASELINE_COLUMNS) != set(baseline_definition_registry_df["baseline_id"]):
    raise ValueError("Baseline column registry does not match the candidate-rule definitions.")


# ---------------------------------------------------------------------
# 3) Export the rule definitions for later benchmark selection
# ---------------------------------------------------------------------

if WRITE_FINAL_OUTPUTS:
    baseline_definition_registry_df.to_parquet(
        BASELINE_DEFINITION_PATH, index=False, compression="zstd"
    )

display(
    baseline_definition_registry_df[
        ["baseline_label", "plain_english_rule", "structure_tested"]
    ].rename(columns={
        "baseline_label": "Candidate rule",
        "plain_english_rule": "Forecasting rule",
        "structure_tested": "What it tests",
    })
)

,Candidate rule,Forecasting rule,What it tests
0,Persistence,Carry the mobility value at the forecast origi...,Immediate level continuity
1,Previous day · same Daypart,Use the target Daypart from exactly one calend...,Daily repetition
2,Previous week · same Daypart,Use the target Daypart from exactly seven cale...,Weekly repetition
3,Recent 7-day Daypart mean,Average the previous seven observations of the...,Smoothed recent Daypart level


**Findings.** The candidate tournament now contains four deliberately simple and distinct forecasting rules. Persistence tests whether the current mobility level simply continues; previous-day and previous-week rules test daily and weekly repetition; and the recent 7-day Daypart mean tests whether smoothing several recent comparable periods provides a better stand-in for the future. None of these rules fits a model or uses the broader forecasting feature package. Their purpose is to establish a transparent forecasting floor from which the Post-CP forecasting and Pre-CP counterfactual benchmark contracts can later be selected.

### Build the target-aligned historical lookup once

The same candidate rules are used across all three modeling tracks, so their historical donors should be resolved once from the authoritative Forecast Sequence history rather than reconstructed independently inside each track.

The lookup preserves the canonical missing rows. If an earlier mobility observation is genuinely missing, the affected candidate forecast remains unavailable rather than jumping over the gap to an older value.

For the recent 7-day Daypart mean, the seven candidate observations are the **same target Daypart on each of the previous seven calendar days**. At least six of those seven observations must be available before the mean is used, matching the comparable-history support convention established upstream.

Most importantly, all donor history is shifted backward before it is summarized. The future target itself can never enter its own baseline prediction.

In [7]:
# ---------------------------------------------------------------------
# 4.2.4.2C — Build the target-aligned candidate history lookup
# ---------------------------------------------------------------------

BASELINE_HISTORY_LOOKUP_PATH = (
    INTERMEDIATE_DIR / "baseline_target_history_lookup.parquet"
)


# ---------------------------------------------------------------------
# 1) Load only the canonical sequence fields required for historical donors
# ---------------------------------------------------------------------
# The Forecast Sequence panel preserves all canonical positions, including
# positions whose observed mobility value is missing. Keeping those rows is
# essential because positional offsets such as 5 and 35 must retain their
# literal one-day and one-week meanings.

sequence_history_df = pd.read_parquet(
    FORECAST_SEQUENCE_PATH,
    columns=[
        "forecast_sequence_id", "observation_sequence_id",
        "metric", "daypart", "observed_value",
    ],
).sort_values(
    ["forecast_sequence_id", "observation_sequence_id"]
).reset_index(drop=True)

history_key = ["forecast_sequence_id", "observation_sequence_id"]

if sequence_history_df.duplicated(history_key).any():
    raise ValueError("Forecast Sequence history is not unique at its canonical key.")


# ---------------------------------------------------------------------
# 2) Build exact previous-day and previous-week target-aligned donors
# ---------------------------------------------------------------------
# Five canonical Dayparts make one calendar day; 35 positions make one week.
# We validate the shifted source IDs so a missing mobility value can never
# compress the sequence and accidentally change the intended calendar lag.

by_sequence = sequence_history_df.groupby("forecast_sequence_id", sort=False)

previous_day_source_id = by_sequence["observation_sequence_id"].shift(STEPS_PER_DAY)
previous_week_source_id = by_sequence["observation_sequence_id"].shift(STEPS_PER_WEEK)

day_source_mask = previous_day_source_id.notna()
week_source_mask = previous_week_source_id.notna()

day_offsets_ok = (
    sequence_history_df.loc[day_source_mask, "observation_sequence_id"]
    .sub(previous_day_source_id.loc[day_source_mask])
    .eq(STEPS_PER_DAY)
    .all()
)

week_offsets_ok = (
    sequence_history_df.loc[week_source_mask, "observation_sequence_id"]
    .sub(previous_week_source_id.loc[week_source_mask])
    .eq(STEPS_PER_WEEK)
    .all()
)

if not day_offsets_ok:
    raise AssertionError("Previous-day donor shifted across a noncanonical sequence gap.")

if not week_offsets_ok:
    raise AssertionError("Previous-week donor shifted across a noncanonical sequence gap.")

sequence_history_df["previous_day_same_daypart_value"] = (
    by_sequence["observed_value"].shift(STEPS_PER_DAY)
)

sequence_history_df["previous_week_same_daypart_value"] = (
    by_sequence["observed_value"].shift(STEPS_PER_WEEK)
)


# ---------------------------------------------------------------------
# 3) Build the shifted recent 7-day same-Daypart mean
# ---------------------------------------------------------------------
# Within Forecast Sequence × Daypart, observations occur once per calendar
# day. shift(1) removes the target day's value BEFORE rolling, preventing the
# future outcome from entering the candidate prediction.

same_daypart = sequence_history_df.groupby(
    ["forecast_sequence_id", "daypart"],
    sort=False,
    observed=True,
)["observed_value"]

sequence_history_df["trailing_7d_same_daypart_mean"] = same_daypart.transform(
    lambda values: values.shift(1).rolling(
        window=TRAILING_HISTORY_DAYS,
        min_periods=TRAILING_MIN_OBS,
    ).mean()
)

sequence_history_df["trailing_7d_same_daypart_obs"] = (
    same_daypart.transform(
        lambda values: values.shift(1).rolling(
            window=TRAILING_HISTORY_DAYS,
            min_periods=1,
        ).count()
    )
    .fillna(0)
    .astype("uint8")
)


# ---------------------------------------------------------------------
# 4) Keep one compact reusable history lookup
# ---------------------------------------------------------------------
# The raw target value is retained here only because persistence joins to an
# origin position later. It is never substituted for the future target being
# scored.

baseline_history_lookup_df = sequence_history_df[
    [
        "forecast_sequence_id", "observation_sequence_id", "metric",
        "observed_value", "previous_day_same_daypart_value",
        "previous_week_same_daypart_value", "trailing_7d_same_daypart_mean",
        "trailing_7d_same_daypart_obs",
    ]
].copy()

if WRITE_INTERMEDIATES:
    baseline_history_lookup_df.to_parquet(
        BASELINE_HISTORY_LOOKUP_PATH, index=False, compression="zstd"
    )


# ---------------------------------------------------------------------
# 5) Summarize structural historical availability
# ---------------------------------------------------------------------
# These percentages cover the full source history, including early periods
# that naturally precede enough lag history. Actual evaluation-period support
# is measured separately after the candidates are attached to forecast rows.

history_lookup_summary_df = (
    baseline_history_lookup_df.groupby("metric", observed=True)
    .agg(
        rows=("observation_sequence_id", "size"),
        observed_value_pct=("observed_value", lambda s: s.notna().mean() * 100),
        previous_day_pct=("previous_day_same_daypart_value", lambda s: s.notna().mean() * 100),
        previous_week_pct=("previous_week_same_daypart_value", lambda s: s.notna().mean() * 100),
        trailing_mean_pct=("trailing_7d_same_daypart_mean", lambda s: s.notna().mean() * 100),
    )
    .reset_index()
)

percentage_columns = [
    "observed_value_pct", "previous_day_pct",
    "previous_week_pct", "trailing_mean_pct",
]

history_lookup_summary_df[percentage_columns] = (
    history_lookup_summary_df[percentage_columns].round(3)
)

display(history_lookup_summary_df)

,metric,rows,observed_value_pct,previous_day_pct,previous_week_pct,trailing_mean_pct
0,fhvhv_avg_trip_speed,1559590,97.684,97.602,97.107,96.860
1,fhvhv_trip_count,1559590,98.199,98.116,97.620,97.278
2,subway_ridership,1559590,58.491,58.491,58.227,58.174
3,taxi_avg_trip_speed,1559590,72.143,72.074,71.658,61.167
4,taxi_trip_count,1559590,98.199,98.116,97.620,97.278


**Findings.** The target-aligned history lookup was constructed successfully across the full canonical sequence without compressing away missing observations. Historical availability is naturally uneven before forecasting eligibility is applied: trip-count history is observed for about 98.2% of canonical positions and FHVHV speed for 97.7%, while Subway ridership and Taxi speed are structurally available across smaller portions of the full citywide sequence universe. Requiring six of seven recent comparable Dayparts makes the trailing mean more demanding, most visibly for Taxi speed, where full-history availability falls to 61.2%. This is expected at this stage because the table includes every canonical Taxi Zone × metric sequence, including series that are not supported for forecasting; the relevant question is whether the already-eligible evaluation populations retain adequate candidate coverage.

### Attach the candidate forecasts to the actual evaluation rows

The history lookup tells us what earlier observation corresponds to each possible target period. The next step attaches those historical values to the **actual rolling-validation and holdout forecasts** already defined in 4.2.3.

The cached prediction files remain wide rather than multiplying each forecast row four times. Each scheduled forecast carries one prediction column for each candidate rule, which preserves the exact forecasting grain while keeping the intermediate files substantially smaller.

Missing target labels are also preserved. A candidate rule may still make a forecast for that scheduled row, but the row cannot contribute to an error score if the actual future mobility value was not observed.

No candidate is fitted here. These remain deterministic historical rules.

In [8]:
# ---------------------------------------------------------------------
# 4.2.4.2D — Materialize the four candidate-rule predictions
# ---------------------------------------------------------------------

BASELINE_MATRIX_COLUMNS = [
    "modeling_track", "stage", "taxi_zone_id", "metric", "forecast_sequence_id",
    "origin_observation_sequence_id", "target_observation_sequence_id", "horizon",
    "target_date", "target_observed_value", "target_label_available",
    "native_track_eligible", "post_cp_common_comparison_eligible",
]

PREDICTION_COLUMNS = list(BASELINE_COLUMNS.values())

BASELINE_SUPPORT_SUMMARY_PATH = (
    INTERMEDIATE_DIR / "baseline_scoring_support_summary.parquet"
)

# Support is summarized while each large prediction surface is already
# in memory. Post-CP and Pre-CP also retain only the compact validation
# columns needed later to select their benchmark contracts.
support_records = []
validation_source_dfs = {}

BENCHMARK_SELECTION_TRACKS = {
    "post_cp_primary",
    "pre_cp_reference",
}

VALIDATION_SOURCE_COLUMNS = [
    "stage", "metric", "forecast_sequence_id", "horizon",
    "target_observed_value", "target_label_available",
    "native_track_eligible", "post_cp_common_comparison_eligible",
    "all_baselines_available", *PREDICTION_COLUMNS,
]

# ---------------------------------------------------------------------
# 1) Prepare compact target-history and origin-history lookups
# ---------------------------------------------------------------------
# Daily, weekly, and trailing-mean rules are aligned to the TARGET period.
# Persistence is the one exception because its definition is simply the
# observed mobility value at the legal forecast origin.

target_history_lookup_df = baseline_history_lookup_df[
    [
        "forecast_sequence_id", "observation_sequence_id", "observed_value",
        "previous_day_same_daypart_value", "previous_week_same_daypart_value",
        "trailing_7d_same_daypart_mean", "trailing_7d_same_daypart_obs",
    ]
].rename(columns={
    "observation_sequence_id": "target_observation_sequence_id",
    "observed_value": "lookup_target_observed_value",
    "previous_day_same_daypart_value": BASELINE_COLUMNS["previous_day_same_daypart"],
    "previous_week_same_daypart_value": BASELINE_COLUMNS["previous_week_same_daypart"],
    "trailing_7d_same_daypart_mean": BASELINE_COLUMNS["trailing_7d_same_daypart_mean"],
})

origin_history_lookup_df = baseline_history_lookup_df[
    ["forecast_sequence_id", "observation_sequence_id", "observed_value"]
].rename(columns={
    "observation_sequence_id": "origin_observation_sequence_id",
    "observed_value": BASELINE_COLUMNS["persistence"],
})


# ---------------------------------------------------------------------
# 2) Build one modeling track at a time
# ---------------------------------------------------------------------
# Processing sequentially prevents the three multi-million-row evaluation
# surfaces from occupying memory simultaneously.

baseline_build_records = []

for track in MODELING_TRACKS:
    stage_filter = (
        [("stage", "==", "rolling_validation")]
        if track == "pre_cp_reference"
        else [("stage", "in", ["rolling_validation", "final_holdout"])]
    )

    # The 86 forecasting predictors are irrelevant to these deterministic
    # candidate rules, so load only identifiers, outcomes, and support flags.
    predictions = pd.read_parquet(
        FINAL_MATRIX_PATHS[track],
        columns=BASELINE_MATRIX_COLUMNS,
        filters=stage_filter,
    )

    # -----------------------------------------------------------------
    # 2a) Attach target-aligned daily, weekly, and trailing-mean donors
    # -----------------------------------------------------------------

    predictions = predictions.merge(
        target_history_lookup_df,
        on=["forecast_sequence_id", "target_observation_sequence_id"],
        how="left",
        validate="many_to_one",
    )

    # -----------------------------------------------------------------
    # 2b) Attach the legal forecast-origin value used by persistence
    # -----------------------------------------------------------------

    predictions = predictions.merge(
        origin_history_lookup_df,
        on=["forecast_sequence_id", "origin_observation_sequence_id"],
        how="left",
        validate="many_to_one",
    )

    # -----------------------------------------------------------------
    # 2c) Record donor positions solely for leakage QA
    # ---------------------------------------------------------------------
    # These IDs are provenance fields, not model inputs. They let the notebook
    # prove that every donor occurs no later than the legal forecast origin.

    predictions["persistence_donor_id"] = predictions["origin_observation_sequence_id"]
    predictions["previous_day_donor_id"] = predictions["target_observation_sequence_id"] - STEPS_PER_DAY
    predictions["previous_week_donor_id"] = predictions["target_observation_sequence_id"] - STEPS_PER_WEEK

    predictions["trailing_latest_donor_id"] = (
        predictions["target_observation_sequence_id"] - STEPS_PER_DAY
    )
    predictions["trailing_earliest_donor_id"] = (
        predictions["target_observation_sequence_id"] - STEPS_PER_WEEK
    )

    # -----------------------------------------------------------------
    # 2d) Verify that the 4.2.3 label matches the authoritative history
    # ---------------------------------------------------------------------
    # lookup_target_observed_value exists only for QA. It is dropped
    # immediately afterward and never becomes a forecasting input.

    matrix_target = predictions["target_observed_value"].to_numpy(dtype="float64", na_value=np.nan)
    lookup_target = predictions["lookup_target_observed_value"].to_numpy(dtype="float64", na_value=np.nan)

    if not np.allclose(matrix_target, lookup_target, equal_nan=True):
        raise AssertionError(
            f"{track}: 4.2.3 target labels disagree with the authoritative sequence history."
        )

    predictions.drop(columns="lookup_target_observed_value", inplace=True)

    # -----------------------------------------------------------------
    # 2e) Enforce the legal information boundary for every donor rule
    # ---------------------------------------------------------------------

    origin_id = predictions["origin_observation_sequence_id"]

    donor_checks = {
        "persistence": predictions["persistence_donor_id"].le(origin_id),
        "previous_day": predictions["previous_day_donor_id"].le(origin_id),
        "previous_week": predictions["previous_week_donor_id"].le(origin_id),
        "trailing_latest": predictions["trailing_latest_donor_id"].le(origin_id),
        "trailing_earliest": predictions["trailing_earliest_donor_id"].le(origin_id),
    }

    failed_donor_checks = [
        name for name, passed in donor_checks.items()
        if not passed.all()
    ]

    if failed_donor_checks:
        raise AssertionError(
            f"{track}: future information entered candidate baseline donors: "
            f"{failed_donor_checks}"
        )

    # At h=5, the origin is exactly one calendar day before the target.
    # Persistence and previous-day same-Daypart therefore MUST coincide.
    h5 = predictions["horizon"].eq(5)

    h5_persistence = predictions.loc[h5, BASELINE_COLUMNS["persistence"]]
    h5_previous_day = predictions.loc[h5, BASELINE_COLUMNS["previous_day_same_daypart"]]

    h5_same = (
        h5_persistence.eq(h5_previous_day)
        | (h5_persistence.isna() & h5_previous_day.isna())
    ).all()

    if not h5_same:
        raise AssertionError(
            f"{track}: h=5 persistence and previous-day candidate forecasts should be identical."
        )

    # -----------------------------------------------------------------
    # 2f) Preserve missing donor history instead of imputing it
    # -----------------------------------------------------------------

    predictions["baseline_available_count"] = (
        predictions[PREDICTION_COLUMNS].notna().sum(axis=1).astype("uint8")
    )

    predictions["all_baselines_available"] = (
        predictions[PREDICTION_COLUMNS].notna().all(axis=1)
    )

    # -----------------------------------------------------------------
    # Measure candidate support while this track is already in memory
    # -----------------------------------------------------------------
    # A forecast can be scored only when the future outcome was observed.
    # Preserve each candidate's individual coverage plus the shared-support
    # population where all four candidate rules can be compared fairly.

    support = predictions[
        [
            "stage", "metric", "horizon", "target_label_available",
            "all_baselines_available", *PREDICTION_COLUMNS,
        ]
    ].copy()

    labeled = (
        support["target_label_available"]
        .fillna(False)
        .astype(bool)
    )

    support["labeled_row"] = labeled

    for baseline_id, prediction_column in BASELINE_COLUMNS.items():
        support[f"scorable__{baseline_id}"] = (
            labeled & support[prediction_column].notna()
        )

    support["scorable__all_four"] = (
        labeled & support["all_baselines_available"]
    )

    grouped = (
        support
        .groupby(["stage", "metric", "horizon"], observed=True)
        .agg(
            label_rows=("labeled_row", "sum"),
            **{
                f"{baseline_id}_rows": (
                    f"scorable__{baseline_id}",
                    "sum",
                )
                for baseline_id in BASELINE_COLUMNS
            },
            all_four_rows=("scorable__all_four", "sum"),
        )
        .reset_index()
    )

    grouped["modeling_track"] = track

    for baseline_id in BASELINE_COLUMNS:
        grouped[f"{baseline_id}_coverage_pct"] = (
            grouped[f"{baseline_id}_rows"]
            / grouped["label_rows"]
            * 100
        )

    grouped["all_four_coverage_pct"] = (
        grouped["all_four_rows"]
        / grouped["label_rows"]
        * 100
    )

    support_records.append(grouped)

    del support, grouped

    # -----------------------------------------------------------------
    # Retain only the validation surfaces that later choose benchmarks
    # -----------------------------------------------------------------
    # Full-History deliberately reuses the Post-CP benchmark, so only
    # Post-CP Primary and Pre-CP Reference need scoring rows retained here.

    if track in BENCHMARK_SELECTION_TRACKS:
        validation_source_dfs[track] = predictions.loc[
            predictions["stage"].eq("rolling_validation"),
            VALIDATION_SOURCE_COLUMNS,
        ].copy()

    if WRITE_INTERMEDIATES:
        predictions.to_parquet(
            BASELINE_PREDICTION_PATHS[track],
            index=False,
            compression="zstd",
        )

    # -----------------------------------------------------------------
    # 2g) Record one compact support summary per Track × Stage
    # ---------------------------------------------------------------------
    # Availability is measured among rows whose future target was observed
    # because only those rows can later contribute to an error metric.

    for stage, stage_df in predictions.groupby("stage", observed=True):
        stage_labeled = stage_df["target_label_available"].fillna(False).astype(bool)
        labeled_rows = int(stage_labeled.sum())

        all_four_pct = (
            stage_df.loc[stage_labeled, "all_baselines_available"].mean() * 100
            if labeled_rows else np.nan
        )

        baseline_build_records.append({
            "modeling_track": TRACK_LABELS[track],
            "stage": str(stage),
            "rows": len(stage_df),
            "labeled_rows": labeled_rows,
            "target_label_coverage_pct": round(stage_labeled.mean() * 100, 3),
            "all_four_available_pct": round(all_four_pct, 3),
        })

    del predictions
    gc.collect()


# ---------------------------------------------------------------------
# 3) Persist the compact support summary assembled during construction
# ---------------------------------------------------------------------

baseline_support_summary_df = pd.concat(
    support_records,
    ignore_index=True,
)

if WRITE_INTERMEDIATES:
    baseline_support_summary_df.to_parquet(
        BASELINE_SUPPORT_SUMMARY_PATH,
        index=False,
        compression="zstd",
    )

baseline_build_summary_df = pd.DataFrame(baseline_build_records)
display(baseline_build_summary_df)

,modeling_track,stage,rows,labeled_rows,target_label_coverage_pct,all_four_available_pct
0,Post-CP Primary,final_holdout,1429320,1419054,99.282,98.749
1,Post-CP Primary,rolling_validation,4570500,4551327,99.581,98.864
2,Full-History Regime-Aware,final_holdout,1355790,1346445,99.311,99.430
3,Full-History Regime-Aware,rolling_validation,4335375,4317777,99.594,99.567
4,Pre-CP Reference,rolling_validation,5704110,5680584,99.588,98.995


**Findings.** The four candidate rules were successfully attached to the actual validation and holdout forecast schedules while preserving the original forecasting grain and legal information boundary. Future target labels remain highly complete across every evaluation surface, ranging from 99.3% to 99.6%, and all four candidate predictions are simultaneously available for roughly 98.7% to 99.6% of labeled rows overall. The much stronger coverage here than in the unrestricted historical lookup confirms that the upstream forecasting-eligibility rules are doing their job: the evaluation experiment is concentrated on Forecast Sequences with sufficient usable history rather than on every structurally possible series.

### Check whether missing history changes the candidate comparison

A fair candidate tournament should not accidentally reward a forecasting rule merely because it fails to make predictions on difficult rows.

Before choosing how the candidate leaderboard will be scored, quantify how often each rule has enough legal historical information to make a prediction.

The most important measure is **“All four”**: the percentage of labeled forecasts for which all four candidate rules can be evaluated against the exact same future outcome.

This is not yet choosing either downstream benchmark. It is checking whether the four candidate rules can compete on a common playing field when we later select the Post-CP forecasting benchmark and the Pre-CP counterfactual benchmark.

In [9]:
# ---------------------------------------------------------------------
# 4.2.4.2E — Measure candidate-rule scoring coverage
# ---------------------------------------------------------------------

# ---------------------------------------------------------------------
# 1) Show the WORST support slice for each Track × Stage
# ---------------------------------------------------------------------
# 2D already measured support while each candidate-prediction surface was
# in memory. Minimum coverage is more informative than average coverage
# because it reveals whether any Target × Horizon loses materially more
# rows in an apples-to-apples candidate comparison.

coverage_columns = [
    "persistence_coverage_pct",
    "previous_day_same_daypart_coverage_pct",
    "previous_week_same_daypart_coverage_pct",
    "trailing_7d_same_daypart_mean_coverage_pct",
    "all_four_coverage_pct",
]

baseline_support_floor_df = (
    baseline_support_summary_df
    .groupby(["modeling_track", "stage"], observed=True)[coverage_columns]
    .min()
    .reset_index()
    .replace({"modeling_track": TRACK_LABELS})
    .rename(columns={
        "modeling_track": "Modeling track",
        "stage": "Stage",
        "persistence_coverage_pct": "Persistence %",
        "previous_day_same_daypart_coverage_pct": "Previous day %",
        "previous_week_same_daypart_coverage_pct": "Previous week %",
        "trailing_7d_same_daypart_mean_coverage_pct": "Recent mean %",
        "all_four_coverage_pct": "All four %",
    })
)

numeric_display_columns = baseline_support_floor_df.columns[2:]

baseline_support_floor_df[numeric_display_columns] = (
    baseline_support_floor_df[numeric_display_columns].round(3)
)

display(baseline_support_floor_df)

,Modeling track,Stage,Persistence %,Previous day %,Previous week %,Recent mean %,All four %
0,Full-History Regime-Aware,final_holdout,99.626,99.647,99.655,99.701,98.838
1,Full-History Regime-Aware,rolling_validation,99.752,99.761,99.768,99.756,99.209
2,Post-CP Primary,final_holdout,98.013,98.242,98.334,97.510,93.807
3,Post-CP Primary,rolling_validation,98.111,98.316,98.359,97.536,94.055
4,Pre-CP Reference,rolling_validation,98.258,98.541,98.522,97.747,94.754


**Findings.** Each individual candidate retains very high scoring support, with the worst Target × Horizon slice still above 97.5% of labeled forecasts. Requiring all four candidates to be available on the same row reduces the worst-case shared support to 94.1% in Post-CP validation and 94.8% in Pre-CP validation; Full-History remains above 99.2% in validation. The gap therefore reflects the intersection of several slightly different missing-history patterns rather than a broadly unsupported candidate rule. This is a modest cost for ensuring that competing candidates are judged against exactly the same future outcomes.

### Visual check — how much validation data can all four candidates score?

The table above is useful for quality assurance, but the practical question is easier to see visually:

> **If all four candidate rules must compete on exactly the same forecasts, how much labeled validation data remains?**

Each square below represents one mobility target and one forecast horizon.

A value near 100% means an apples-to-apples comparison costs almost no usable validation data. Lower values would warn us that requiring common support removes a meaningful part of the forecasting experiment.

Only the two **benchmark-selection experiments** appear here. Full-History does not choose a separate benchmark, so adding it would not change the methodological decision.

In [10]:
# ---------------------------------------------------------------------
# 4.2.4.2F — Visualize common candidate-rule scoring support
# ---------------------------------------------------------------------

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from project_branding import BRAND_COLORS, apply_branding


# ---------------------------------------------------------------------
# 1) Show only the experiments that actually SELECT a benchmark contract
# ---------------------------------------------------------------------
# Post-CP Primary selects the Post-CP forecasting benchmark.
# Pre-CP Reference separately selects the counterfactual benchmark.
# Full-History deliberately reuses the Post-CP forecasting benchmark.

visual_tracks = [
    ("post_cp_primary", "Post-CP forecasting benchmark selection"),
    ("pre_cp_reference", "Pre-CP counterfactual benchmark selection"),
]

target_order = FORECAST_TARGETS
horizon_order = FORECAST_HORIZONS
horizon_labels = [f"h={h}" for h in horizon_order]

# Stack the experiments vertically because both use the exact same horizon
# dimension. This gives the reader one shared x-axis instead of two copies.
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=[label for _, label in visual_tracks],
    vertical_spacing=0.16,
)

coverage_scale = [
    [0.00, BRAND_COLORS["terracotta"]],
    [0.50, BRAND_COLORS["pale_peach"]],
    [0.80, BRAND_COLORS["seafoam"]],
    [1.00, BRAND_COLORS["dark_teal"]],
]


# ---------------------------------------------------------------------
# 2) Draw one Target × Horizon heatmap per benchmark-selection experiment
# ---------------------------------------------------------------------

for row, (track, _) in enumerate(visual_tracks, start=1):
    panel = baseline_support_summary_df.loc[
        baseline_support_summary_df["modeling_track"].eq(track)
        & baseline_support_summary_df["stage"].eq("rolling_validation")
    ].copy()

    pivot = (
        panel.pivot(index="metric", columns="horizon", values="all_four_coverage_pct")
        .reindex(index=target_order, columns=horizon_order)
    )

    z = pivot.to_numpy(dtype=float)

    # Keep visible labels to one decimal so the chart stays easy to scan.
    text = [
        ["—" if pd.isna(value) else f"{value:.1f}%" for value in values]
        for values in z
    ]

    fig.add_trace(
        go.Heatmap(
            z=z,
            x=horizon_labels,
            y=[TARGET_LABELS[target] for target in target_order],
            text=text,
            texttemplate="%{text}",
            zmin=0,
            zmax=100,
            colorscale=coverage_scale,
            hovertemplate=(
                "<b>%{y}</b><br>"
                "%{x}<br>"
                "All four candidates available: %{z:.1f}%"
                "<extra></extra>"
            ),
            # Both panels use the same 0–100% scale; show one legend only.
            showscale=(row == 1),
            colorbar=dict(
                title="Usable<br>validation %",
                len=0.78,
                y=0.50,
            ) if row == 1 else None,
        ),
        row=row, col=1,
    )


# ---------------------------------------------------------------------
# 3) Use one shared forecast-horizon axis for both experiments
# ---------------------------------------------------------------------

# Hide the redundant upper tick labels; the lower axis represents both panels.
fig.update_xaxes(showticklabels=False, row=1, col=1)
fig.update_xaxes(
    title_text="Forecast horizon",
    tickmode="array",
    tickvals=horizon_labels,
    ticktext=horizon_labels,
    row=2,
    col=1,
)

fig.update_layout(
    title={
        "text": (
            "How much validation data can all four candidate rules score?"
            "<br><sup>Higher means an apples-to-apples tournament costs less evaluation coverage.</sup>"
        ),
        "x": 0.02,
    },
    width=940,
    height=650,
    margin=dict(l=150, r=100, t=95, b=65),
)

fig = apply_branding(fig)
fig.show()

**Findings.** The heatmap shows that the shared-support reduction is highly concentrated rather than widespread. For the Post-CP benchmark-selection experiment, Taxi trips, FHVHV trips, FHVHV speed, and Subway ridership retain approximately 99.6–99.8% common validation support across all three horizons; Taxi average speed is the exception, retaining about 94.1–95.8%. The Pre-CP counterfactual experiment shows the same pattern: most targets retain roughly 99.0–99.6%, while Taxi average speed retains about 94.8–96.3%. Because even the weakest cells preserve roughly 19 out of every 20 labeled forecasts, a shared-support tournament can provide a substantially cleaner apples-to-apples comparison without materially shrinking the experiment.

> ✅ Decision — Use a Shared\-Support Candidate Leaderboard
How should the four candidate rules be compared when their historical coverage is not perfectly identical?

We will score the four candidate rules only on labeled forecast rows where all four candidates can make a prediction\.

This keeps the tournament apples\-to\-apples: every candidate's MAE will be calculated against the same future outcomes, so a rule cannot appear artificially stronger simply because it happens to be unavailable on more difficult forecasts\.

The executed support check shows that this fairness constraint costs relatively little data\. In rolling validation, the weakest shared\-support slice still retains about 94\.1% of labeled Post\-CP forecasts and 94\.8% of labeled Pre\-CP forecasts, while most Target × Horizon combinations retain roughly 99–100%\. The modest reduction is concentrated mainly in Taxi average speed rather than spread across the forecasting system\.

We will therefore:

\* use the shared\-support validation population to compare the four candidate rules and select each Target × Horizon winner;
\* continue reporting each candidate's individual coverage so any missing\-history differences remain visible;
\* select the Post\-CP forecasting benchmark from Post\-CP Primary validation;
\* select the Pre\-CP counterfactual benchmark separately from Pre\-CP Reference validation; and
\* keep holdout results completely out of benchmark selection\.

This sacrifices a small amount of evaluation coverage in exchange for a substantially cleaner and more defensible benchmark comparison\.

### Section Summary

**Section Summary.** Four simple candidate rules are now fully defined and materialized using target-aligned historical information that was legally available at each original forecast origin. Candidate coverage is extremely high within the supported forecasting populations, and the small loss from requiring all four rules on the same evaluation row is concentrated mainly in Taxi average speed rather than across the forecasting system. No candidate performance has yet been used to choose a winner, and no holdout error has influenced selection. The experiment is therefore ready to compare the four candidates on a common validation population and select the Post-CP forecasting benchmark and Pre-CP counterfactual benchmark independently.

## 4.2.4.3 — Compare the candidate rules on validation data

We now know that all four candidate rules can be scored on nearly the same set of forecasts. The next question is simple:

> **Which of these very simple rules makes the smallest average mistake?**

We will answer that question using **validation data only**. The final holdout remains untouched while the benchmark is being chosen.

The main score is **MAE — mean absolute error**. In plain language, MAE is the average size of a forecast's miss, ignoring whether the forecast was too high or too low. The candidate with the **lowest MAE** is the better forecast.

We will also keep several supporting measures because each tells us something different:

- **RMSE:** similar to MAE, but large misses hurt more.
- **NMAE:** puts the error in context by comparing it with the overall size of the values being forecast.
- **Bias:** shows whether forecasts tend to run too high or too low.
- **Skill vs. persistence:** shows how much error a rule removes compared with simply carrying the current value forward.
- **Forecasts and Forecast Sequences scored:** show how much evidence sits behind each result.

Only **MAE chooses the winner**. The other measures help us understand the result; they do not get a vote.

The Post-CP forecasting benchmark is tested on the shared Post-CP/Full-History comparison population so the benchmark used by those two model tracks is built on the same series they will later share. The Pre-CP counterfactual benchmark is tested separately on its legal 2024 validation period.

In [11]:
# ---------------------------------------------------------------------
# 4.2.4.3A — Build the two validation scoring populations
# ---------------------------------------------------------------------

# ---------------------------------------------------------------------
# 1) Define the two experiments that actually choose a benchmark
# ---------------------------------------------------------------------
# Post-CP Primary chooses the benchmark later shared with Full-History.
# Pre-CP Reference chooses a separate benchmark for the counterfactual.
#
# Both selections use validation rows only. Holdout data is deliberately
# absent from this block so it cannot influence which candidate wins.

VALIDATION_EXPERIMENTS = {
    "post_cp_forecasting": {
        "track": "post_cp_primary",
        "label": "Post-CP forecasting benchmark",
        "population_flag": "post_cp_common_comparison_eligible",
    },
    "pre_cp_counterfactual": {
        "track": "pre_cp_reference",
        "label": "Pre-CP counterfactual benchmark",
        "population_flag": "native_track_eligible",
    },
}


# ---------------------------------------------------------------------
# 2) Build one shared-support validation population per experiment
# ---------------------------------------------------------------------
# A row is scored only when:
#   • it belongs to rolling validation;
#   • it belongs to the correct Forecast Sequence population;
#   • the future outcome was observed; and
#   • all four candidate rules can make a prediction.
#
# This applies the shared-support decision made in the previous section.

validation_population_dfs = {}
validation_population_records = []

for experiment_id, spec in VALIDATION_EXPERIMENTS.items():
    source = validation_source_dfs[spec["track"]]

    population_ok = source[spec["population_flag"]].fillna(False).astype(bool)
    label_ok = source["target_label_available"].fillna(False).astype(bool)
    shared_support_ok = source["all_baselines_available"].fillna(False).astype(bool)

    scored = source.loc[
        population_ok & label_ok & shared_support_ok
    ].copy()

    if scored.empty:
        raise ValueError(f"{experiment_id}: no shared-support validation rows remain.")

    expected_horizons = set(FORECAST_HORIZONS)
    observed_horizons = set(scored["horizon"].dropna().astype(int).unique())

    if observed_horizons != expected_horizons:
        raise AssertionError(
            f"{experiment_id}: validation horizons are {sorted(observed_horizons)}, "
            f"expected {sorted(expected_horizons)}."
        )

    validation_population_dfs[experiment_id] = scored

    validation_population_records.append({
        "Benchmark being selected": spec["label"],
        "Validation forecasts": len(scored),
        "Forecast Sequences": scored["forecast_sequence_id"].nunique(),
        "Targets": scored["metric"].nunique(),
        "Horizons": ", ".join(f"h={h}" for h in sorted(observed_horizons)),
    })

# ---------------------------------------------------------------------
# 3) Show exactly what evidence will be used to choose each benchmark
# ---------------------------------------------------------------------

# The filtered shared-support populations are authoritative from here forward.
# Release the larger pre-filter validation surfaces before candidate scoring.
del validation_source_dfs
gc.collect()

validation_population_summary_df = pd.DataFrame(validation_population_records)
display(validation_population_summary_df)

,Benchmark being selected,Validation forecasts,Forecast Sequences,Targets,Horizons
0,Post-CP forecasting benchmark,4277785,1042,5,"h=1, h=2, h=5"
1,Pre-CP counterfactual benchmark,5623474,1039,5,"h=1, h=2, h=5"


**Findings.** The two benchmark-selection experiments retain large, well-supported validation populations after applying the shared-support rule. The Post-CP forecasting experiment contains **4,277,785 forecasts across 1,042 Forecast Sequences**, while the Pre-CP counterfactual experiment contains **5,623,474 forecasts across 1,039 Forecast Sequences**. Both retain all five targets and h=1, h=2, and h=5. Because every scored row has an observed future value and predictions from all four candidate rules, each candidate is compared against exactly the same outcomes, and no holdout results can influence benchmark selection.

### Score every candidate on the same forecasts

Each candidate is now measured against the exact same future outcomes within a given Target × Horizon.

That matters because we are no longer asking whether a rule *can* make a forecast. We are asking which rule makes the **smaller mistakes when both are asked to predict the same thing**.

For every candidate we calculate the average miss, the effect of unusually large misses, whether it tends to overshoot or undershoot, and how it compares with persistence.

A positive **skill vs. persistence** number is especially useful for interpretation:

- **+20%** means the candidate reduced average error by 20% compared with carrying the current value forward.
- **0%** means it performed the same as persistence.
- **−20%** means it produced 20% more error than persistence.

This number helps explain the results, but **lowest MAE remains the sole rule used to choose a benchmark**.

In [12]:
# ---------------------------------------------------------------------
# 4.2.4.3B — Score the candidate rules
# ---------------------------------------------------------------------

def summarize_candidate(
    source: pd.DataFrame,
    *,
    experiment_id: str,
    baseline_id: str,
    prediction_column: str,
) -> pd.DataFrame:
    """Summarize one candidate rule by Target × Horizon on shared rows."""

    # The shared-support population guarantees a prediction and an observed
    # outcome for every candidate, so all candidates use identical rows.
    work = source[
        [
            "metric", "horizon", "forecast_sequence_id",
            "target_observed_value", prediction_column,
        ]
    ].copy()

    work["error"] = work[prediction_column] - work["target_observed_value"]
    work["absolute_error"] = work["error"].abs()
    work["squared_error"] = work["error"].pow(2)
    work["absolute_actual"] = work["target_observed_value"].abs()

    grouped = (
        work.groupby(["metric", "horizon"], observed=True)
        .agg(
            evaluated_rows=("error", "size"),
            supported_forecast_sequences=("forecast_sequence_id", "nunique"),
            mae=("absolute_error", "mean"),
            mean_squared_error=("squared_error", "mean"),
            absolute_error_sum=("absolute_error", "sum"),
            absolute_actual_sum=("absolute_actual", "sum"),
            bias=("error", "mean"),
        )
        .reset_index()
    )

    grouped["rmse"] = np.sqrt(grouped.pop("mean_squared_error"))

    # NMAE answers: "How large was the total error compared with the total
    # size of what we were forecasting?" A zero denominator remains missing.
    grouped["nmae"] = np.where(
        grouped["absolute_actual_sum"].gt(0),
        grouped["absolute_error_sum"] / grouped["absolute_actual_sum"],
        np.nan,
    )

    grouped.drop(
        columns=["absolute_error_sum", "absolute_actual_sum"],
        inplace=True,
    )

    grouped["experiment_id"] = experiment_id
    grouped["baseline_id"] = baseline_id
    return grouped


# ---------------------------------------------------------------------
# 1) Score all four candidates in both benchmark-selection experiments
# ---------------------------------------------------------------------

candidate_score_frames = []

for experiment_id, validation_df in validation_population_dfs.items():
    for baseline_id, prediction_column in BASELINE_COLUMNS.items():
        candidate_score_frames.append(
            summarize_candidate(
                validation_df,
                experiment_id=experiment_id,
                baseline_id=baseline_id,
                prediction_column=prediction_column,
            )
        )

baseline_leaderboard_df = pd.concat(candidate_score_frames, ignore_index=True)

baseline_leaderboard_df = baseline_leaderboard_df.merge(
    baseline_definition_registry_df[["baseline_id", "baseline_label"]],
    on="baseline_id",
    how="left",
    validate="many_to_one",
)


# ---------------------------------------------------------------------
# 2) Compare each candidate with persistence on the same Target × Horizon
# ---------------------------------------------------------------------
# Positive skill means fewer average errors than persistence.
# Persistence itself is therefore exactly 0% by definition.

persistence_mae_df = (
    baseline_leaderboard_df.loc[
        baseline_leaderboard_df["baseline_id"].eq("persistence"),
        ["experiment_id", "metric", "horizon", "mae"],
    ]
    .rename(columns={"mae": "persistence_mae"})
)

baseline_leaderboard_df = baseline_leaderboard_df.merge(
    persistence_mae_df,
    on=["experiment_id", "metric", "horizon"],
    how="left",
    validate="many_to_one",
)

baseline_leaderboard_df["skill_vs_persistence"] = np.where(
    baseline_leaderboard_df["persistence_mae"].gt(0),
    1 - baseline_leaderboard_df["mae"] / baseline_leaderboard_df["persistence_mae"],
    np.nan,
)

baseline_leaderboard_df["skill_vs_persistence_pct"] = (
    baseline_leaderboard_df["skill_vs_persistence"] * 100
)


# ---------------------------------------------------------------------
# 3) Rank candidates using MAE only
# ---------------------------------------------------------------------
# The rank is descriptive for now. We do NOT freeze a winner in this block.
# Exact-tie handling is deliberately settled before the winner registry is made.

baseline_leaderboard_df["mae_rank"] = (
    baseline_leaderboard_df
    .groupby(["experiment_id", "metric", "horizon"], observed=True)["mae"]
    .rank(method="min", ascending=True)
    .astype(int)
)

column_order = [
    "experiment_id", "metric", "horizon", "baseline_id", "baseline_label",
    "mae_rank", "mae", "rmse", "nmae", "bias",
    "skill_vs_persistence", "skill_vs_persistence_pct",
    "evaluated_rows", "supported_forecast_sequences", "persistence_mae",
]

baseline_leaderboard_df = (
    baseline_leaderboard_df[column_order]
    .sort_values(["experiment_id", "metric", "horizon", "mae_rank", "baseline_id"])
    .reset_index(drop=True)
)

if WRITE_FINAL_OUTPUTS:
    baseline_leaderboard_df.to_parquet(
        BASELINE_LEADERBOARD_PATH, index=False, compression="zstd"
    )


# ---------------------------------------------------------------------
# 4) Show the two strongest candidates for every Target × Horizon
# ---------------------------------------------------------------------
# Raw MAE is meaningful only within the same target because trip counts,
# ridership, and speeds live on very different numerical scales.

top_two_validation_df = (
    baseline_leaderboard_df.loc[baseline_leaderboard_df["mae_rank"].le(2)]
    .assign(
        Benchmark=lambda d: d["experiment_id"].map({
            key: value["label"] for key, value in VALIDATION_EXPERIMENTS.items()
        }),
        Target=lambda d: d["metric"].map(TARGET_LABELS),
        Candidate=lambda d: d["baseline_label"],
        MAE=lambda d: d["mae"].round(3),
        **{"Skill vs persistence": lambda d: d["skill_vs_persistence_pct"].round(1)},
    )
    [
        [
            "Benchmark", "Target", "horizon", "mae_rank",
            "Candidate", "MAE", "Skill vs persistence",
        ]
    ]
    .rename(columns={
        "horizon": "Horizon",
        "mae_rank": "MAE rank",
    })
)

display(top_two_validation_df)

,Benchmark,Target,Horizon,MAE rank,Candidate,MAE,Skill vs persistence
0,Post-CP forecasting benchmark,FHVHV average speed,1,1,Previous week · same Daypart,1.305,63.0
1,Post-CP forecasting benchmark,FHVHV average speed,1,2,Recent 7-day Daypart mean,1.523,56.9
4,Post-CP forecasting benchmark,FHVHV average speed,2,1,Previous week · same Daypart,1.306,74.4
5,Post-CP forecasting benchmark,FHVHV average speed,2,2,Recent 7-day Daypart mean,1.524,70.1
8,Post-CP forecasting benchmark,FHVHV average speed,5,1,Previous week · same Daypart,1.309,21.6
...,...,...,...,...,...,...,...
112,Pre-CP counterfactual benchmark,Taxi trips,2,1,Previous week · same Daypart,16.804,79.7
113,Pre-CP counterfactual benchmark,Taxi trips,2,2,Previous day · same Daypart,22.247,73.2
116,Pre-CP counterfactual benchmark,Taxi trips,5,1,Previous week · same Daypart,16.777,24.5
117,Pre-CP counterfactual benchmark,Taxi trips,5,2,Persistence,22.211,0.0


**Findings.** The validation tournament produces a remarkably consistent result. **Previous week · same Daypart** has the lowest MAE in **27 of the 30** Benchmark × Target × Horizon comparisons, including all 15 Post-CP combinations. The only exceptions are Pre-CP Taxi average speed at h=1, h=2, and h=5, where the **Recent 7-day Daypart mean** performs best. Persistence and previous-day same-Daypart do not win any benchmark slot, and every selected winner improves on persistence. The simple benchmark contract is therefore clear rather than dependent on small or inconsistent differences between candidate rules.


> ✅ Decision — Freeze the forecasting benchmark contract
Select the rank\-1 validation\-MAE candidate independently for each Benchmark × Target × Horizon\.

\* Post\-CP forecasting benchmark: use Previous week · same Daypart for all 15 Target × Horizon combinations\.
\* Pre\-CP counterfactual benchmark: use Previous week · same Daypart for 12 of 15 combinations; use Recent 7\-day Daypart mean for Taxi average speed at h = 1, 2, and 5\.

Previous day · same Daypart and Persistence are not selected for any benchmark cell\. Every selected winner has positive skill relative to persistence\.

The validation tournament therefore supports freezing the benchmark choices now and carrying them unchanged into subsequent model comparison\. Holdout data remains reserved for final evaluation\.

### Does a simple recurring pattern beat “just carry the current value forward”?

Raw MAE tells us which candidate makes the smallest mistakes, but its scale changes from one target to another. A Taxi trip-count error and a speed error cannot be compared directly.

For us, the clearer question is:

> **How much error can the best simple historical pattern remove compared with persistence?**

The next chart looks only at the three candidates that use a recurring historical pattern — previous day, previous week, or the recent 7-day average — and finds the strongest one for each Target × Horizon.

A positive number means some simple recurring pattern beats persistence. A negative number means persistence is still better than all three structured alternatives.

This does **not** decide which advanced model to use. It simply shows how much useful short-term repetition is already visible before we introduce more complicated forecasting methods.

In [13]:
# ---------------------------------------------------------------------
# 4.2.4.3C — Visualize simple forecastability
# ---------------------------------------------------------------------

# ---------------------------------------------------------------------
# 1) Find the strongest NON-persistence candidate for each Target × Horizon
# ---------------------------------------------------------------------
# We intentionally exclude persistence here because the chart asks whether
# any simple daily/weekly/recent-history pattern improves on that basic floor.

structured_candidates_df = baseline_leaderboard_df.loc[
    ~baseline_leaderboard_df["baseline_id"].eq("persistence")
].copy()

best_structured_idx = (
    structured_candidates_df
    .groupby(["experiment_id", "metric", "horizon"], observed=True)["mae"]
    .idxmin()
)

best_structured_df = (
    structured_candidates_df.loc[best_structured_idx]
    .sort_values(["experiment_id", "metric", "horizon"])
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------
# 2) Draw the two benchmark-selection experiments on one shared horizon axis
# ---------------------------------------------------------------------

visual_experiments = [
    ("post_cp_forecasting", "Post-CP forecasting benchmark"),
    ("pre_cp_counterfactual", "Pre-CP counterfactual benchmark"),
]

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=[label for _, label in visual_experiments],
    vertical_spacing=0.16,
)

# A diverging scale makes the meaning immediate:
# terracotta = worse than persistence, light = similar, teal = better.
skill_scale = [
    [0.00, BRAND_COLORS["terracotta"]],
    [0.50, BRAND_COLORS["ice"]],
    [1.00, BRAND_COLORS["dark_teal"]],
]

# Use the same symmetric scale across both panels so the colors mean the
# same thing everywhere. Cap the visible range only after seeing all values.
max_abs_skill = best_structured_df["skill_vs_persistence_pct"].abs().max()
skill_limit = max(5.0, float(np.ceil(max_abs_skill / 5.0) * 5.0))

for row, (experiment_id, _) in enumerate(visual_experiments, start=1):
    panel = best_structured_df.loc[
        best_structured_df["experiment_id"].eq(experiment_id)
    ].copy()

    skill_pivot = (
        panel.pivot(
            index="metric",
            columns="horizon",
            values="skill_vs_persistence_pct",
        )
        .reindex(index=FORECAST_TARGETS, columns=FORECAST_HORIZONS)
    )

    candidate_pivot = (
        panel.pivot(
            index="metric",
            columns="horizon",
            values="baseline_label",
        )
        .reindex(index=FORECAST_TARGETS, columns=FORECAST_HORIZONS)
    )

    z = skill_pivot.to_numpy(dtype=float)

    text = [
        ["—" if pd.isna(value) else f"{value:+.1f}%" for value in values]
        for values in z
    ]

    customdata = candidate_pivot.to_numpy(dtype=object)

    fig.add_trace(
        go.Heatmap(
            z=z,
            x=[f"h={h}" for h in FORECAST_HORIZONS],
            y=[TARGET_LABELS[target] for target in FORECAST_TARGETS],
            text=text,
            texttemplate="%{text}",
            customdata=customdata,
            zmin=-skill_limit,
            zmax=skill_limit,
            zmid=0,
            colorscale=skill_scale,
            hovertemplate=(
                "<b>%{y}</b><br>"
                "%{x}<br>"
                "Best simple pattern: %{customdata}<br>"
                "Error change vs persistence: %{z:+.1f}%"
                "<extra></extra>"
            ),
            showscale=(row == 1),
            colorbar=dict(
                title="Error removed<br>vs persistence",
                ticksuffix="%",
                len=0.78,
                y=0.50,
            ) if row == 1 else None,
        ),
        row=row, col=1,
    )

# One horizon axis serves both experiments.
fig.update_xaxes(showticklabels=False, row=1, col=1)
fig.update_xaxes(title_text="Forecast horizon", row=2, col=1)

fig.update_layout(
    title={
        "text": (
            "Can a simple recurring pattern beat carrying the current value forward?"
            "<br><sup>Positive values mean less average error than persistence; negative values mean more.</sup>"
        ),
        "x": 0.02,
    },
    width=940,
    height=650,
    margin=dict(l=150, r=110, t=100, b=65),
)

fig = apply_branding(fig)
fig.show()

**Findings.** A simple recurring historical pattern beats persistence for **every target and horizon in both validation experiments**. In the Post-CP experiment, the strongest structured rule reduces MAE by about **15.1% to 89.9%** relative to persistence; in the Pre-CP experiment, the reduction ranges from about **18.3% to 89.7%**. Subway ridership shows the largest gains, approaching **90% less error than persistence at h=2**. Previous-week repetition is strongest almost everywhere, with the recent 7-day mean replacing it only for Pre-CP Taxi average speed. The lower relative gains at h=5 should not be read as the structured benchmark suddenly becoming much less accurate: at h=5, persistence itself becomes a stronger comparison because carrying the origin forward is equivalent to using the previous day's same Daypart. The structured benchmark's own error remains nearly unchanged across these short horizons.

### Check whether the improvement is widespread or driven by a few Forecast Sequences

An overall MAE can improve even if the better result is concentrated in only a small number of places.

To make sure the headline score is representative, we also compare each candidate with persistence **one Forecast Sequence at a time**.

For example, if a candidate beats persistence in 85% of the supported Forecast Sequences, its advantage is widespread. If it beats persistence in only 20%, an overall improvement may be coming from a much smaller group.

This check does not choose the benchmark either. It simply tells us how broadly the apparent improvement is shared.

In [14]:
# ---------------------------------------------------------------------
# 4.2.4.3D — Measure how broadly candidates beat persistence
# ---------------------------------------------------------------------

BASELINE_SEQUENCE_PERFORMANCE_PATH = (
    FINAL_DIR / "baseline_sequence_performance.parquet"
)


# ---------------------------------------------------------------------
# 1) Calculate MAE separately for each Forecast Sequence
# ---------------------------------------------------------------------

sequence_score_frames = []

for experiment_id, source in validation_population_dfs.items():
    for baseline_id, prediction_column in BASELINE_COLUMNS.items():
        work = source[
            [
                "metric", "horizon", "forecast_sequence_id",
                "target_observed_value", prediction_column,
            ]
        ].copy()

        work["absolute_error"] = (
            work[prediction_column] - work["target_observed_value"]
        ).abs()

        sequence_scores = (
            work.groupby(
                ["metric", "horizon", "forecast_sequence_id"],
                observed=True,
            )
            .agg(
                mae=("absolute_error", "mean"),
                evaluated_rows=("absolute_error", "size"),
            )
            .reset_index()
        )

        sequence_scores["experiment_id"] = experiment_id
        sequence_scores["baseline_id"] = baseline_id
        sequence_score_frames.append(sequence_scores)

baseline_sequence_performance_df = pd.concat(
    sequence_score_frames, ignore_index=True
)

baseline_sequence_performance_df = baseline_sequence_performance_df.merge(
    baseline_definition_registry_df[["baseline_id", "baseline_label"]],
    on="baseline_id",
    how="left",
    validate="many_to_one",
)


# ---------------------------------------------------------------------
# 2) Put each sequence's persistence MAE beside every candidate
# ---------------------------------------------------------------------

sequence_persistence_df = (
    baseline_sequence_performance_df.loc[
        baseline_sequence_performance_df["baseline_id"].eq("persistence"),
        ["experiment_id", "metric", "horizon", "forecast_sequence_id", "mae"],
    ]
    .rename(columns={"mae": "persistence_mae"})
)

baseline_sequence_performance_df = baseline_sequence_performance_df.merge(
    sequence_persistence_df,
    on=["experiment_id", "metric", "horizon", "forecast_sequence_id"],
    how="left",
    validate="many_to_one",
)

baseline_sequence_performance_df["beats_persistence"] = (
    baseline_sequence_performance_df["mae"]
    < baseline_sequence_performance_df["persistence_mae"]
)

baseline_sequence_performance_df["ties_persistence"] = np.isclose(
    baseline_sequence_performance_df["mae"],
    baseline_sequence_performance_df["persistence_mae"],
    rtol=1e-12,
    atol=1e-12,
)

if WRITE_FINAL_OUTPUTS:
    baseline_sequence_performance_df.to_parquet(
        BASELINE_SEQUENCE_PERFORMANCE_PATH,
        index=False,
        compression="zstd",
    )


# ---------------------------------------------------------------------
# 3) Summarize how widespread each candidate's advantage is
# ---------------------------------------------------------------------

sequence_breadth_df = (
    baseline_sequence_performance_df
    .groupby(
        ["experiment_id", "metric", "horizon", "baseline_id", "baseline_label"],
        observed=True,
    )
    .agg(
        forecast_sequences=("forecast_sequence_id", "nunique"),
        sequences_beating_persistence=("beats_persistence", "sum"),
        sequences_tying_persistence=("ties_persistence", "sum"),
    )
    .reset_index()
)

sequence_breadth_df["share_beating_persistence_pct"] = (
    sequence_breadth_df["sequences_beating_persistence"]
    / sequence_breadth_df["forecast_sequences"]
    * 100
)

sequence_breadth_df["share_tying_persistence_pct"] = (
    sequence_breadth_df["sequences_tying_persistence"]
    / sequence_breadth_df["forecast_sequences"]
    * 100
)


# ---------------------------------------------------------------------
# 4) Show breadth for the strongest structured candidate in each slot
# ---------------------------------------------------------------------

best_structured_breadth_df = (
    best_structured_df[
        ["experiment_id", "metric", "horizon", "baseline_id"]
    ]
    .merge(
        sequence_breadth_df,
        on=["experiment_id", "metric", "horizon", "baseline_id"],
        how="left",
        validate="one_to_one",
    )
    .assign(
        Benchmark=lambda d: d["experiment_id"].map({
            key: value["label"] for key, value in VALIDATION_EXPERIMENTS.items()
        }),
        Target=lambda d: d["metric"].map(TARGET_LABELS),
        Candidate=lambda d: d["baseline_label"],
        **{
            "Sequences beating persistence": lambda d:
                d["share_beating_persistence_pct"].round(1)
        },
    )
    [
        [
            "Benchmark", "Target", "horizon", "Candidate",
            "forecast_sequences", "Sequences beating persistence",
        ]
    ]
    .rename(columns={
        "horizon": "Horizon",
        "forecast_sequences": "Forecast Sequences",
    })
)

display(best_structured_breadth_df)

,Benchmark,Target,Horizon,Candidate,Forecast Sequences,Sequences beating persistence
0,Post-CP forecasting benchmark,FHVHV average speed,1,Previous week · same Daypart,256,100.0
1,Post-CP forecasting benchmark,FHVHV average speed,2,Previous week · same Daypart,256,100.0
2,Post-CP forecasting benchmark,FHVHV average speed,5,Previous week · same Daypart,256,99.6
3,Post-CP forecasting benchmark,FHVHV trips,1,Previous week · same Daypart,257,99.6
4,Post-CP forecasting benchmark,FHVHV trips,2,Previous week · same Daypart,257,99.6
5,Post-CP forecasting benchmark,FHVHV trips,5,Previous week · same Daypart,257,97.7
6,Post-CP forecasting benchmark,Subway ridership,1,Previous week · same Daypart,149,100.0
7,Post-CP forecasting benchmark,Subway ridership,2,Previous week · same Daypart,149,100.0
8,Post-CP forecasting benchmark,Subway ridership,5,Previous week · same Daypart,149,98.7
9,Post-CP forecasting benchmark,Taxi average speed,1,Previous week · same Daypart,123,100.0


**Findings.** The improvement over persistence is widespread across Forecast Sequences rather than being driven by a small number of unusually easy series. For the strongest structured rule in each Target × Horizon, the share of individual sequences beating persistence ranges from **87.2% to 100%** in the Post-CP experiment and from **79.5% to 100%** in the Pre-CP experiment. Most FHVHV speed, FHVHV trip, Subway ridership, and Taxi speed combinations are at or near universal improvement. Taxi trips at h=5 are the least uniform case, but the previous-week rule still beats persistence for **87.2%** of Post-CP sequences and **79.5%** of Pre-CP sequences. The overall advantage is therefore broad enough to treat the selected benchmarks as meaningful forecasting floors rather than averages driven by a handful of zones.

> ✅ Decision — Resolve the exact\-tie rule
When candidate baselines have exactly equal validation MAE on the same forecast population, break the tie using the fixed simplicity order:

Persistence → Previous day · same Daypart → Previous week · same Daypart → Recent 7\-day Daypart mean

Apply this rule only to exact MAE ties\. Any strictly lower MAE remains the winner regardless of the size of the difference\.

This keeps validation MAE as the sole performance criterion and prevents RMSE, bias, holdout performance, or another post hoc metric from changing benchmark selection\.

### Section Summary

Four simple forecasting rules were compared on shared rolling-validation populations using **MAE as the only winner-selection score**, with the final holdout kept completely out of the decision. Previous week · same Daypart wins **27 of 30** benchmark slots, including every Post-CP Target × Horizon; the only exceptions are the three Pre-CP Taxi average-speed horizons, where the recent 7-day Daypart mean performs best. Every selected rule beats persistence, with MAE reductions ranging from roughly **15% to 90%**, and those gains are broad across individual Forecast Sequences rather than concentrated in a few places. The benchmark choices are therefore clear enough to freeze: previous-week repetition is the default forecasting floor, with the recent 7-day mean used only for Pre-CP Taxi speed. Exact ties can remain governed by the fixed simplicity rule, although no winning benchmark slot required that rule in this run.

## 4.2.4.4 — Establish the Benchmark for Advanced Models

The validation tournament has now done its job.

For every Benchmark × Target × Horizon, we know which simple historical rule produces the lowest validation MAE. We have also fixed the only ambiguity in winner selection: an exact MAE tie is resolved using the precommitted simplicity order rather than introducing another performance metric.

The next step is therefore not another model-selection exercise. It is to turn those validation results into a **frozen benchmark contract** that downstream forecasting models must beat.

Two rules remain important:

- the **Post-CP forecasting benchmark** is selected once from the Post-CP validation experiment and is reused unchanged by both Post-CP Primary and Full-History Regime-Aware;
- the **Pre-CP counterfactual benchmark** remains separate because it represents a different historical forecasting problem.

The final holdout still plays no role here. These benchmark identities are frozen entirely from rolling-validation evidence.

In [15]:
# ---------------------------------------------------------------------
# 4.2.4.4A — Freeze benchmark winners
# ---------------------------------------------------------------------

# ---------------------------------------------------------------------
# 1) Apply the locked MAE winner rule and exact-tie simplicity order
# ---------------------------------------------------------------------
# All required scoring objects were built earlier in this notebook.
# The benchmark contract is selected from validation results only.

tie_order = [
    "persistence",
    "previous_day_same_daypart",
    "previous_week_same_daypart",
    "trailing_7d_same_daypart_mean",
]
tie_rank = {baseline_id: rank for rank, baseline_id in enumerate(tie_order)}

winner_records = []
tie_rule_text = (
    "Minimum validation MAE; exact ties within rtol=1e-12 and atol=1e-12 "
    "resolved by precommitted simplicity order."
)

for (experiment_id, metric, horizon), group in baseline_leaderboard_df.groupby(
    ["experiment_id", "metric", "horizon"], observed=True
):
    minimum_mae = float(group["mae"].min())

    tied = group.loc[
        np.isclose(group["mae"], minimum_mae, rtol=1e-12, atol=1e-12)
    ].copy()

    tied["simplicity_rank"] = tied["baseline_id"].map(tie_rank)

    winner = tied.sort_values(
        ["simplicity_rank", "baseline_id"],
        kind="stable",
    ).iloc[0].copy()

    winner_records.append({
        "experiment_id": experiment_id,
        "benchmark_label": VALIDATION_EXPERIMENTS[experiment_id]["label"],
        "metric": metric,
        "horizon": int(horizon),
        "baseline_id": winner["baseline_id"],
        "baseline_label": winner["baseline_label"],
        "mae": winner["mae"],
        "rmse": winner["rmse"],
        "nmae": winner["nmae"],
        "bias": winner["bias"],
        "skill_vs_persistence": winner["skill_vs_persistence"],
        "skill_vs_persistence_pct": winner["skill_vs_persistence_pct"],
        "evaluated_rows": winner["evaluated_rows"],
        "supported_forecast_sequences": winner["supported_forecast_sequences"],
        "candidates_tied_at_minimum_mae": int(len(tied)),
        "selection_stage": "rolling_validation",
        "selection_metric": "mae",
        "tie_break_rule": tie_rule_text,
    })


# ---------------------------------------------------------------------
# 2) Freeze one benchmark per Benchmark × Target × Horizon
# ---------------------------------------------------------------------

frozen_benchmark_contract_df = (
    pd.DataFrame(winner_records)
    .sort_values(["experiment_id", "metric", "horizon"])
    .reset_index(drop=True)
)

FROZEN_BENCHMARK_CONTRACT_PATH = (
    FINAL_DIR / "frozen_benchmark_contract.parquet"
)

if WRITE_FINAL_OUTPUTS:
    frozen_benchmark_contract_df.to_parquet(
        FROZEN_BENCHMARK_CONTRACT_PATH,
        index=False,
        compression="zstd",
    )


# ---------------------------------------------------------------------
# 3) Validate the expected winner pattern
# ---------------------------------------------------------------------

assert len(frozen_benchmark_contract_df) == 30
assert not frozen_benchmark_contract_df.duplicated(
    ["experiment_id", "metric", "horizon"]
).any()
assert set(frozen_benchmark_contract_df["horizon"]) == set(FORECAST_HORIZONS)

post_winners = frozen_benchmark_contract_df.loc[
    frozen_benchmark_contract_df["experiment_id"].eq("post_cp_forecasting")
]

assert post_winners["baseline_id"].eq(
    "previous_week_same_daypart"
).all()

pre_winners = frozen_benchmark_contract_df.loc[
    frozen_benchmark_contract_df["experiment_id"].eq("pre_cp_counterfactual")
]

taxi_speed_id = "taxi_avg_trip_speed"

assert pre_winners.loc[
    pre_winners["metric"].eq(taxi_speed_id),
    "baseline_id",
].eq("trailing_7d_same_daypart_mean").all()

assert pre_winners.loc[
    ~pre_winners["metric"].eq(taxi_speed_id),
    "baseline_id",
].eq("previous_week_same_daypart").all()


# ---------------------------------------------------------------------
# 4) Show the frozen benchmark contract
# ---------------------------------------------------------------------

frozen_benchmark_display_df = frozen_benchmark_contract_df.assign(
    Benchmark=lambda d: d["benchmark_label"],
    Target=lambda d: d["metric"].map(TARGET_LABELS),
    Horizon=lambda d: d["horizon"],
    **{
        "Frozen benchmark": lambda d: d["baseline_label"],
        "Validation MAE": lambda d: d["mae"].round(3),
        "Skill vs persistence %": lambda d: d["skill_vs_persistence_pct"].round(1),
        "Candidates tied at minimum MAE": lambda d:
            d["candidates_tied_at_minimum_mae"],
    },
)[[
    "Benchmark", "Target", "Horizon", "Frozen benchmark",
    "Validation MAE", "Skill vs persistence %",
    "Candidates tied at minimum MAE",
]]

display(frozen_benchmark_display_df)

,Benchmark,Target,Horizon,Frozen benchmark,Validation MAE,Skill vs persistence %,Candidates tied at minimum MAE
0,Post-CP forecasting benchmark,FHVHV average speed,1,Previous week · same Daypart,1.305,63.0,1
1,Post-CP forecasting benchmark,FHVHV average speed,2,Previous week · same Daypart,1.306,74.4,1
2,Post-CP forecasting benchmark,FHVHV average speed,5,Previous week · same Daypart,1.309,21.6,1
3,Post-CP forecasting benchmark,FHVHV trips,1,Previous week · same Daypart,69.782,67.3,1
4,Post-CP forecasting benchmark,FHVHV trips,2,Previous week · same Daypart,69.782,76.0,1
5,Post-CP forecasting benchmark,FHVHV trips,5,Previous week · same Daypart,69.726,35.2,1
6,Post-CP forecasting benchmark,Subway ridership,1,Previous week · same Daypart,542.189,84.3,1
7,Post-CP forecasting benchmark,Subway ridership,2,Previous week · same Daypart,542.141,89.9,1
8,Post-CP forecasting benchmark,Subway ridership,5,Previous week · same Daypart,542.150,48.3,1
9,Post-CP forecasting benchmark,Taxi average speed,1,Previous week · same Daypart,2.268,50.2,1


**Findings.** The benchmark-selection results now resolve to one unambiguous rule for each of the 30 Benchmark × Target × Horizon combinations. Previous week · same Daypart is frozen for all 15 Post-CP combinations and for 12 of the 15 Pre-CP combinations. The only departures are Pre-CP Taxi average speed at h=1, h=2, and h=5, where Recent 7-day Daypart mean has the lowest validation MAE. No benchmark identity is selected from holdout performance, and the fixed simplicity order is invoked only if multiple candidates are numerically tied at the minimum MAE.

### Carry the frozen benchmark into the downstream modeling tracks

The benchmark contract describes two forecasting problems, but the next modeling notebooks operate on three modeling tracks.

That distinction is deliberate.

**Post-CP Primary** and **Full-History Regime-Aware** answer the same post-CP forecasting question. Full-History is allowed to use more historical information when fitting its models, but it is not allowed to choose an easier benchmark. Both tracks therefore inherit the same Post-CP benchmark for each Target × Horizon.

**Pre-CP Reference** inherits the separately selected Pre-CP counterfactual benchmark.

Making that mapping explicit now prevents later model families from silently redefining their reference forecast.

In [16]:
# 4.2.4.4B — Build the advanced-model benchmark registry

experiment_to_tracks = {
    "post_cp_forecasting": ["post_cp_primary", "full_history_regime_aware"],
    "pre_cp_counterfactual": ["pre_cp_reference"],
}

registry_records = []
for experiment_id, track_ids in experiment_to_tracks.items():
    winners = frozen_benchmark_contract_df.loc[
        frozen_benchmark_contract_df["experiment_id"].eq(experiment_id)
    ].copy()
    for track_id in track_ids:
        track_winners = winners.copy()
        track_winners["modeling_track"] = track_id
        track_winners["modeling_track_label"] = TRACK_LABELS[track_id]
        registry_records.append(track_winners)

advanced_model_benchmark_registry_df = (
    pd.concat(registry_records, ignore_index=True)
    .sort_values(["modeling_track", "metric", "horizon"])
    .reset_index(drop=True)
)

if WRITE_FINAL_OUTPUTS:
    advanced_model_benchmark_registry_df.to_parquet(
        FROZEN_BENCHMARK_PATH, index=False, compression="zstd"
    )

assert len(advanced_model_benchmark_registry_df) == 45
assert not advanced_model_benchmark_registry_df.duplicated(
    ["modeling_track", "metric", "horizon"]
).any()

post_registry = advanced_model_benchmark_registry_df.loc[
    advanced_model_benchmark_registry_df["modeling_track"].isin(
        ["post_cp_primary", "full_history_regime_aware"]
    ),
    ["modeling_track", "metric", "horizon", "baseline_id"],
].copy()

post_wide = post_registry.pivot(
    index=["metric", "horizon"],
    columns="modeling_track",
    values="baseline_id",
)
assert post_wide.shape == (15, 2)
assert post_wide["post_cp_primary"].eq(
    post_wide["full_history_regime_aware"]
).all()

display_df = advanced_model_benchmark_registry_df[[
    "modeling_track_label", "metric", "horizon", "baseline_label"
]].copy()
display_df.columns = ["Modeling track", "Target", "Horizon", "Benchmark"]
display_df["Target"] = display_df["Target"].map(TARGET_LABELS)
display(display_df)

,Modeling track,Target,Horizon,Benchmark
0,Full-History Regime-Aware,FHVHV average speed,1,Previous week · same Daypart
1,Full-History Regime-Aware,FHVHV average speed,2,Previous week · same Daypart
2,Full-History Regime-Aware,FHVHV average speed,5,Previous week · same Daypart
3,Full-History Regime-Aware,FHVHV trips,1,Previous week · same Daypart
4,Full-History Regime-Aware,FHVHV trips,2,Previous week · same Daypart
5,Full-History Regime-Aware,FHVHV trips,5,Previous week · same Daypart
6,Full-History Regime-Aware,Subway ridership,1,Previous week · same Daypart
7,Full-History Regime-Aware,Subway ridership,2,Previous week · same Daypart
8,Full-History Regime-Aware,Subway ridership,5,Previous week · same Daypart
9,Full-History Regime-Aware,Taxi average speed,1,Previous week · same Daypart


**Findings.** The frozen benchmark choices have now been turned into **45 explicit Modeling Track × Target × Horizon assignments** for downstream model evaluation. Post-CP Primary and Full-History Regime-Aware receive the **same Post-CP forecasting benchmark in all 15 Target × Horizon slots**, so the Full-History models will not be allowed to compete against an easier reference forecast. Pre-CP Reference receives its own 15 counterfactual benchmark assignments, including the three Taxi average-speed horizons where the **Recent 7-day Daypart mean** won validation instead of **Previous week · same Daypart**. Every downstream forecasting slot therefore has exactly one fixed benchmark before any advanced model is trained.

### Summarize what the simple benchmarks tell us about forecastability

Selecting a benchmark and deciding whether a forecasting problem contains useful structure are related but not identical questions.

The winner registry tells us **which simple rule advanced models must beat**.

Forecastability asks a broader empirical question:

> **Do these mobility series contain recurring historical structure that predicts future observations better than simply carrying the current value forward?**

We will not impose an arbitrary pass/fail threshold. Instead, each Target × Horizon is described using the evidence already produced by the rolling-validation experiment:

- the frozen benchmark's validation MAE and normalized MAE;
- its percentage improvement relative to persistence;
- the share of Forecast Sequences on which it beats persistence; and
- how its error changes as the forecast horizon moves from h=1 to h=2 to h=5.

These measures describe the forecasting difficulty without changing the benchmark winner.

In [17]:
# ---------------------------------------------------------------------
# 4.2.4.4C — Build the empirical forecastability summary
# ---------------------------------------------------------------------

# ---------------------------------------------------------------------
# 1) Attach the sequence-level breadth evidence to each frozen winner
# ---------------------------------------------------------------------
# sequence_breadth_df was built earlier from the same shared-support
# validation experiment. Reuse that in-memory result directly.

frozen_scores_df = frozen_benchmark_contract_df.merge(
    sequence_breadth_df,
    on=["experiment_id", "metric", "horizon", "baseline_id"],
    how="left",
    validate="one_to_one",
    suffixes=("", "_breadth"),
)


# ---------------------------------------------------------------------
# 2) Use each target's h=1 benchmark error as its short-horizon reference
# ---------------------------------------------------------------------

h1_reference_df = (
    frozen_scores_df.loc[
        frozen_scores_df["horizon"].eq(1),
        ["experiment_id", "metric", "mae"],
    ]
    .rename(columns={"mae": "h1_benchmark_mae"})
)

empirical_forecastability_df = frozen_scores_df.merge(
    h1_reference_df,
    on=["experiment_id", "metric"],
    how="left",
    validate="many_to_one",
)

empirical_forecastability_df["mae_change_vs_h1_pct"] = np.where(
    empirical_forecastability_df["h1_benchmark_mae"].ne(0),
    (
        empirical_forecastability_df["mae"]
        / empirical_forecastability_df["h1_benchmark_mae"]
        - 1
    ) * 100,
    np.nan,
)


# ---------------------------------------------------------------------
# 3) Keep one compact forecastability handoff table
# ---------------------------------------------------------------------

forecastability_columns = [
    "experiment_id", "benchmark_label", "metric", "horizon",
    "baseline_id", "baseline_label", "mae", "rmse", "nmae", "bias",
    "skill_vs_persistence_pct", "forecast_sequences",
    "sequences_beating_persistence", "share_beating_persistence_pct",
    "share_tying_persistence_pct", "h1_benchmark_mae",
    "mae_change_vs_h1_pct", "evaluated_rows",
    "supported_forecast_sequences",
]

empirical_forecastability_df = (
    empirical_forecastability_df[forecastability_columns]
    .sort_values(["experiment_id", "metric", "horizon"])
    .reset_index(drop=True)
)

if WRITE_FINAL_OUTPUTS:
    empirical_forecastability_df.to_parquet(
        FORECASTABILITY_SUMMARY_PATH,
        index=False,
        compression="zstd",
    )

assert len(empirical_forecastability_df) == 30

assert empirical_forecastability_df.loc[
    empirical_forecastability_df["horizon"].eq(1),
    "mae_change_vs_h1_pct",
].abs().max() < 1e-12


# ---------------------------------------------------------------------
# 4) Show the reader-facing forecastability summary
# ---------------------------------------------------------------------

display(
    empirical_forecastability_df.assign(
        Benchmark=lambda d: d["benchmark_label"],
        Target=lambda d: d["metric"].map(TARGET_LABELS),
        Horizon=lambda d: d["horizon"],
        **{
            "Frozen benchmark": lambda d: d["baseline_label"],
            "MAE": lambda d: d["mae"].round(3),
            "NMAE": lambda d: d["nmae"].round(3),
            "Skill vs persistence %": lambda d:
                d["skill_vs_persistence_pct"].round(1),
            "Sequences beating persistence %": lambda d:
                d["share_beating_persistence_pct"].round(1),
            "MAE change vs h=1 %": lambda d:
                d["mae_change_vs_h1_pct"].round(1),
        },
    )[[
        "Benchmark", "Target", "Horizon", "Frozen benchmark", "MAE",
        "NMAE", "Skill vs persistence %",
        "Sequences beating persistence %", "MAE change vs h=1 %",
    ]]
)

,Benchmark,Target,Horizon,Frozen benchmark,MAE,NMAE,Skill vs persistence %,Sequences beating persistence %,MAE change vs h=1 %
0,Post-CP forecasting benchmark,FHVHV average speed,1,Previous week · same Daypart,1.305,0.077,63.0,100.0,0.0
1,Post-CP forecasting benchmark,FHVHV average speed,2,Previous week · same Daypart,1.306,0.078,74.4,100.0,0.1
2,Post-CP forecasting benchmark,FHVHV average speed,5,Previous week · same Daypart,1.309,0.078,21.6,99.6,0.3
3,Post-CP forecasting benchmark,FHVHV trips,1,Previous week · same Daypart,69.782,0.134,67.3,99.6,0.0
4,Post-CP forecasting benchmark,FHVHV trips,2,Previous week · same Daypart,69.782,0.134,76.0,99.6,-0.0
5,Post-CP forecasting benchmark,FHVHV trips,5,Previous week · same Daypart,69.726,0.134,35.2,97.7,-0.1
6,Post-CP forecasting benchmark,Subway ridership,1,Previous week · same Daypart,542.189,0.112,84.3,100.0,0.0
7,Post-CP forecasting benchmark,Subway ridership,2,Previous week · same Daypart,542.141,0.112,89.9,100.0,-0.0
8,Post-CP forecasting benchmark,Subway ridership,5,Previous week · same Daypart,542.150,0.112,48.3,98.7,-0.0
9,Post-CP forecasting benchmark,Taxi average speed,1,Previous week · same Daypart,2.268,0.168,50.2,100.0,0.0


**Findings.** The forecastability summary shows that the frozen benchmarks provide a meaningful forecasting floor across **all five targets and all three horizons**. Every selected benchmark has positive validation skill relative to persistence, and most of those gains are shared across a large majority of Forecast Sequences rather than being produced by a small number of unusually easy series. The strength of the advantage varies by target and horizon: Subway ridership shows the largest reductions in error, while Taxi trips at h=5 are the least uniform case, where the benchmark still beats persistence for **87.2% of Post-CP sequences and 79.5% of Pre-CP sequences**. The table also shows that the benchmark's own error remains nearly flat from h=1 through h=5, so the forecasting problem does not become materially less stable over these short horizons. Together, these results support carrying the complete five-target, three-horizon scope forward and asking whether richer models can improve on an already competitive simple benchmark.

> **How stable is simple benchmark error across forecast horizons?**

A forecasting problem may contain strong short-range structure but become much harder as the target moves farther into the future.

Because the five mobility targets have very different numerical scales, raw MAE cannot be placed on one common axis. Instead, the next view expresses each frozen benchmark's MAE relative to its own h=1 error.

A value of:

- **0%** means benchmark error is essentially unchanged from h=1;
- **+25%** means average error is 25% larger than at h=1;
- **−10%** means average error is 10% smaller than at h=1.

This is descriptive evidence about horizon sensitivity. It does not alter the frozen benchmark rule.

In [18]:
# 4.2.4.4D — Visualize horizon decay

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from project_branding import BRAND_COLORS, apply_branding

if RUN_VISUALS:
    horizon_fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        subplot_titles=[
            "Post-CP forecasting benchmark",
            "Pre-CP counterfactual benchmark",
        ],
        vertical_spacing=0.16,
    )

    for row, experiment_id in enumerate(
        ["post_cp_forecasting", "pre_cp_counterfactual"], start=1
    ):
        panel = empirical_forecastability_df.loc[
            empirical_forecastability_df["experiment_id"].eq(experiment_id)
        ].copy()

        for metric in FORECAST_TARGETS:
            line = panel.loc[panel["metric"].eq(metric)].sort_values("horizon")
            horizon_fig.add_trace(
                go.Scatter(
                    x=[f"h={h}" for h in FORECAST_HORIZONS],
                    y=line["mae_change_vs_h1_pct"],
                    mode="lines+markers",
                    name=TARGET_LABELS[metric],
                    legendgroup=metric,
                    showlegend=(row == 1),
                    hovertemplate=(
                        "<b>%{fullData.name}</b><br>"
                        "%{x}<br>"
                        "MAE change vs h=1: %{y:.1f}%"
                        "<extra></extra>"
                    ),
                ),
                row=row,
                col=1,
            )

    horizon_fig.add_hline(y=0, line_dash="dash", line_color=BRAND_COLORS["dark_teal"])
    horizon_fig.update_yaxes(title_text="MAE change vs h=1 (%)")
    horizon_fig.update_xaxes(title_text="Forecast horizon", row=2, col=1)
    horizon_fig.update_layout(
        title={
            "text": (
                "How does frozen-benchmark error change as the forecast moves farther ahead?"
                "<br><sup>Each line is normalized to its own h=1 MAE.</sup>"
            ),
            "x": 0.02,
        },
        width=940,
        height=760,
        margin=dict(l=120, r=40, t=110, b=65),
    )
    horizon_fig = apply_branding(horizon_fig)
    horizon_fig.show()

**Findings.** The frozen simple benchmarks are strikingly stable across the three short forecast horizons. Relative to h=1, benchmark MAE changes by only about **−0.1% to +0.4%** for every Post-CP target and by about **−0.1% to +1.0%** for every Pre-CP target. In other words, the selected weekly or recent-average rule itself does not meaningfully deteriorate as the forecast moves from h=1 to h=5. This stability is partly a consequence of the target-aligned baseline design: the previous-week rule always uses the same comparable period one week before the target, regardless of whether the forecast is issued one, two, or five Dayparts ahead. The much larger changes in skill relative to persistence therefore mostly reflect how the persistence comparison changes with horizon rather than a collapse in the selected benchmark's own accuracy.


> ✅ Conclusion — Carry the Full Forecasting Scope Forward
The baseline results support continuing with all five forecasting targets at h=1, h=2, and h=5\.

Every Target × Horizon has a valid benchmark, every selected benchmark improves on persistence, and the gains are broad across the supported Forecast Sequences\. The selected benchmarks also remain stable across the three short forecast horizons, so there is no evidence that any target or horizon should be removed before the richer model families are tested\.

### Section Summary

The simple-baseline experiment has now produced the forecasting floor that the later models must beat. **Previous week · same Daypart** is the Post-CP benchmark for every target and horizon and is also the Pre-CP benchmark everywhere except Taxi average speed, where the **Recent 7-day Daypart mean** performs better.

More importantly, the baseline results give us no reason to narrow the forecasting problem before testing richer models. Every selected benchmark beats persistence, those gains appear across a large majority of Forecast Sequences, and benchmark error stays nearly unchanged from h=1 through h=5.

The full scope therefore carries forward: **five targets × three horizons**, with separate Post-CP, Full-History, and Pre-CP modeling tracks and frozen benchmark rules already attached to each one.

## 4.2.4.5 — Package the Supported Modeling Scope

The baseline experiment answered the question it was built to answer: **is there enough repeatable structure in these mobility series to justify trying more capable forecasting methods?**

For all five targets and all three horizons, the answer is yes.

The next forecasting stages will test three broader approaches:

- **Classical time-series models:** models that learn from ordered mobility history, with some versions able to add approved outside context.
- **Panel tree models:** models that can use the broader leakage-safe feature matrix and learn nonlinear patterns across many Forecast Sequences together.
- **Neural sequence models:** models that learn from ordered historical windows while also using approved forecasting context.

This section does **not** choose a winning model family. It simply records that all five targets and all three horizons remain in scope and attaches the correct frozen benchmark to every downstream modeling job.

In [19]:
# ---------------------------------------------------------------------
# 4.2.4.5A — Build the supported downstream modeling scope
# ---------------------------------------------------------------------

SUPPORTED_MODELING_SCOPE_PATH = (
    FINAL_DIR / "supported_modeling_scope.parquet"
)


# ---------------------------------------------------------------------
# 1) Define the three downstream model families
# ---------------------------------------------------------------------
# These are scope assignments only. They do not claim that every family
# will ultimately outperform the frozen benchmark.

DOWNSTREAM_MODEL_FAMILIES = [
    {
        "chapter": "4.3",
        "model_family": "classical_time_series",
        "model_family_label": "Classical time-series models",
        "modeling_role": (
            "Learn from ordered mobility history, with approved time-safe "
            "context added where the model supports it."
        ),
    },
    {
        "chapter": "4.4",
        "model_family": "panel_tree",
        "model_family_label": "Panel tree models",
        "modeling_role": (
            "Use the broader leakage-safe forecasting matrix and learn "
            "nonlinear relationships across Forecast Sequences."
        ),
    },
    {
        "chapter": "4.5",
        "model_family": "neural_sequence",
        "model_family_label": "Neural sequence models",
        "modeling_role": (
            "Learn from ordered historical windows with approved dynamic, "
            "calendar, spatial, static, and weather context."
        ),
    },
]

model_family_df = pd.DataFrame(DOWNSTREAM_MODEL_FAMILIES)


# ---------------------------------------------------------------------
# 2) Expand each frozen Track × Target × Horizon benchmark into the
#    three downstream model families
# ---------------------------------------------------------------------

supported_modeling_scope_df = (
    advanced_model_benchmark_registry_df.assign(_join_key=1)
    .merge(
        model_family_df.assign(_join_key=1),
        on="_join_key",
        how="inner",
    )
    .drop(columns="_join_key")
)

supported_modeling_scope_df["scope_status"] = "continue"
supported_modeling_scope_df["benchmark_selection_metric"] = "mae"
supported_modeling_scope_df["benchmark_selection_stage"] = "rolling_validation"

supported_modeling_scope_df["evaluation_note"] = np.where(
    supported_modeling_scope_df["modeling_track"].isin(
        ["post_cp_primary", "full_history_regime_aware"]
    ),
    (
        "Preserve the native modeling population; use the shared Post-CP / "
        "Full-History cohort only when comparing those tracks directly."
    ),
    (
        "Use the native Pre-CP Reference population; this track has no "
        "ordinary post-CP holdout."
    ),
)

supported_modeling_scope_df["counterfactual_role"] = np.where(
    supported_modeling_scope_df["modeling_track"].eq("pre_cp_reference"),
    "Candidate model family for the later frozen no-CP counterfactual",
    "Ordinary forecasting model track",
)

scope_columns = [
    "chapter", "model_family", "model_family_label", "modeling_role",
    "modeling_track", "modeling_track_label", "metric", "horizon",
    "scope_status", "baseline_id", "baseline_label", "benchmark_label",
    "benchmark_selection_metric", "benchmark_selection_stage",
    "evaluation_note", "counterfactual_role",
]

supported_modeling_scope_df = (
    supported_modeling_scope_df[scope_columns]
    .sort_values(["chapter", "modeling_track", "metric", "horizon"])
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------
# 3) Validate and persist the downstream scope
# ---------------------------------------------------------------------

expected_scope_rows = (
    len(DOWNSTREAM_MODEL_FAMILIES)
    * len(MODELING_TRACKS)
    * len(FORECAST_TARGETS)
    * len(FORECAST_HORIZONS)
)

assert len(supported_modeling_scope_df) == expected_scope_rows == 135

assert not supported_modeling_scope_df.duplicated(
    ["model_family", "modeling_track", "metric", "horizon"]
).any()

assert supported_modeling_scope_df["scope_status"].eq("continue").all()

if WRITE_FINAL_OUTPUTS:
    supported_modeling_scope_df.to_parquet(
        SUPPORTED_MODELING_SCOPE_PATH,
        index=False,
        compression="zstd",
    )


# ---------------------------------------------------------------------
# 4) Show the compact downstream handoff
# ---------------------------------------------------------------------

supported_scope_summary_df = (
    supported_modeling_scope_df
    .groupby(
        ["chapter", "model_family_label", "modeling_track_label"],
        observed=True,
    )
    .agg(
        Targets=("metric", "nunique"),
        Horizons=("horizon", "nunique"),
        **{"Target × Horizon jobs": ("metric", "size")},
    )
    .reset_index()
    .rename(columns={
        "chapter": "Next section",
        "model_family_label": "Model family",
        "modeling_track_label": "Modeling track",
    })
)

display(supported_scope_summary_df)

,Next section,Model family,Modeling track,Targets,Horizons,Target × Horizon jobs
0,4.3,Classical time-series models,Full-History Regime-Aware,5,3,15
1,4.3,Classical time-series models,Post-CP Primary,5,3,15
2,4.3,Classical time-series models,Pre-CP Reference,5,3,15
3,4.4,Panel tree models,Full-History Regime-Aware,5,3,15
4,4.4,Panel tree models,Post-CP Primary,5,3,15
5,4.4,Panel tree models,Pre-CP Reference,5,3,15
6,4.5,Neural sequence models,Full-History Regime-Aware,5,3,15
7,4.5,Neural sequence models,Post-CP Primary,5,3,15
8,4.5,Neural sequence models,Pre-CP Reference,5,3,15


**Findings.** The downstream modeling scope is complete and balanced. Each of the three modeling families receives all **five targets at h=1, h=2, and h=5** for each of the three modeling tracks, giving **15 Target × Horizon jobs per track** and **135 assignments overall**. No target, horizon, or modeling track was dropped during packaging. The later classical, panel-tree, and neural stages therefore inherit the full forecasting scope together with the benchmark contract already assigned to each job.

### What the handoff means for the next forecasting notebooks

The supported-scope registry deliberately keeps **what we will test** separate from **what ultimately works best**.

Every downstream family gets the opportunity to forecast all five targets at h=1, h=2, and h=5. That does not mean every family is expected to win every job. It means the baseline evidence gave us no good reason to remove any forecasting problem before richer models get a chance to improve on it.

The frozen benchmarks also keep the comparison fair:

- Post-CP Primary and Full-History Regime-Aware must beat the **same Post-CP forecasting benchmark**;
- Pre-CP Reference is judged against its separately selected **Pre-CP counterfactual benchmark**;
- validation remains the place where model settings are chosen;
- the final Post-CP holdout remains untouched while models are being selected; and
- the Pre-CP track remains the source of the later no-congestion-pricing forecast.

The richer models therefore receive **more information and more flexibility, but not an easier benchmark**.

### Section Summary

All **five targets × three horizons × three modeling tracks** remain supported for the next forecasting stages. Each of those 45 forecasting jobs has now been mapped into the classical, panel-tree, and neural model families, producing a complete downstream scope without changing native track populations or frozen benchmark rules.

The next models therefore inherit a clear job: use their additional modeling power to improve on a simple historical pattern that has already proven difficult to beat with another simple rule.

## 4.2.4.6 — Final Baseline QA and Handoff

Close the baseline stage by publishing the forecasting evidence required by downstream models and validating the full handoff.

The final package includes both compact benchmark contracts and the row-level **Pre-CP Reference benchmark surface** required for Chapter 5. That surface contains the native-track-eligible 2024 rolling-validation rows with observed targets, including genuine observed zeros, and preserves rows even when the selected benchmark itself is unavailable.

Two support definitions remain explicit:

* **baseline tournament support** identifies rows where all four simple baseline candidates were available and could therefore be compared on a common denominator;

* **selected-benchmark support** identifies rows where the frozen benchmark for that Target × Horizon is available for downstream comparison or fallback.

The final QA verifies benchmark selection, forecastability, modeling scope, and the row-level Pre-CP benchmark handoff before publishing the final manifest.

**4.2.4.6A — Publish the selected Pre-CP benchmark predictions.**

Chapter 5 requires the frozen Pre-CP benchmark at the same row-level forecast grain used by advanced models.

This cell publishes one stable benchmark prediction for every native-track-eligible Pre-CP Reference rolling-validation row with an observed target. The output retains the physical forecast identity, observed target, selected benchmark identity, benchmark prediction, and explicit support flags.

`baseline_tournament_support_eligible` identifies rows where all four simple baseline candidates were available and therefore belonged to the common denominator used for benchmark selection.

`selected_benchmark_support_eligible` identifies rows where the selected benchmark itself is available and can therefore serve as the comparison or fallback forecast in downstream model evaluation.

The final artifact contains only the selected benchmark required downstream; candidate-specific baseline prediction columns remain in the detailed baseline-evaluation surface.


In [20]:
# ---------------------------------------------------------------------
# 4.2.4.6A — Publish the selected Pre-CP benchmark predictions
# ---------------------------------------------------------------------

# ---------------------------------------------------------------------
# 1) Load the compact benchmark and timing contracts
# ---------------------------------------------------------------------
# The benchmark registry determines which rule applies to each Target ×
# Horizon. The track contract fixes the legal 2024 validation window.

frozen_benchmark_contract_path = (
    FINAL_DIR / "frozen_benchmark_contract.parquet"
)

advanced_model_benchmark_registry_df = pd.read_parquet(
    FROZEN_BENCHMARK_PATH
)

frozen_benchmark_contract_df = pd.read_parquet(
    frozen_benchmark_contract_path
)

track_contract_df = pd.read_parquet(
    TRACK_CONTRACT_PATH,
    columns=[
        "modeling_track",
        "rolling_validation_target_start_date",
        "rolling_validation_target_end_date",
    ],
)

for column in [
    "rolling_validation_target_start_date",
    "rolling_validation_target_end_date",
]:
    track_contract_df[column] = pd.to_datetime(
        track_contract_df[column]
    ).dt.normalize()


# ---------------------------------------------------------------------
# 2) Isolate the frozen 15-job Pre-CP benchmark roster
# ---------------------------------------------------------------------

pre_cp_benchmark_registry_df = (
    advanced_model_benchmark_registry_df.loc[
        advanced_model_benchmark_registry_df["modeling_track"].eq(
            "pre_cp_reference"
        ),
        [
            "modeling_track",
            "metric",
            "horizon",
            "baseline_id",
            "baseline_label",
            "selection_metric",
            "selection_stage",
        ],
    ]
    .copy()
    .sort_values(["metric", "horizon"])
    .reset_index(drop=True)
)

if len(pre_cp_benchmark_registry_df) != 15:
    raise AssertionError(
        "Expected exactly 15 Pre-CP Target × Horizon benchmarks; "
        f"found {len(pre_cp_benchmark_registry_df):,}."
    )

if pre_cp_benchmark_registry_df.duplicated(
    ["modeling_track", "metric", "horizon"]
).any():
    raise AssertionError(
        "Pre-CP benchmark registry is not unique by Target × Horizon."
    )

if not pre_cp_benchmark_registry_df["selection_stage"].eq(
    "rolling_validation"
).all():
    raise AssertionError(
        "Pre-CP benchmark registry contains an unexpected selection stage."
    )

if not pre_cp_benchmark_registry_df["selection_metric"].eq("mae").all():
    raise AssertionError(
        "Pre-CP benchmark registry contains an unexpected selection metric."
    )


# ---------------------------------------------------------------------
# 3) Resolve the frozen Pre-CP validation window
# ---------------------------------------------------------------------

pre_cp_track_contract_df = track_contract_df.loc[
    track_contract_df["modeling_track"].eq("pre_cp_reference")
].copy()

if len(pre_cp_track_contract_df) != 1:
    raise AssertionError(
        "Expected exactly one Pre-CP Reference track-contract row."
    )

validation_start = pre_cp_track_contract_df.iloc[0][
    "rolling_validation_target_start_date"
]

validation_end = pre_cp_track_contract_df.iloc[0][
    "rolling_validation_target_end_date"
]


# ---------------------------------------------------------------------
# 4) Load only the source columns needed by the final handoff
# ---------------------------------------------------------------------
# Only benchmark rules that actually appear in the frozen Pre-CP roster
# need to be read from the four-candidate intermediate prediction surface.

selected_baseline_ids = sorted(
    pre_cp_benchmark_registry_df["baseline_id"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

selected_prediction_columns = [
    f"baseline__{baseline_id}"
    for baseline_id in selected_baseline_ids
]

source_columns = [
    "modeling_track",
    "stage",
    "taxi_zone_id",
    "metric",
    "forecast_sequence_id",
    "origin_observation_sequence_id",
    "target_observation_sequence_id",
    "horizon",
    "target_date",
    "target_observed_value",
    "target_label_available",
    "native_track_eligible",
    "all_baselines_available",
    *selected_prediction_columns,
]

pre_cp_source_path = BASELINE_PREDICTION_PATHS["pre_cp_reference"]

if not pre_cp_source_path.exists():
    raise FileNotFoundError(
        f"Missing Pre-CP baseline prediction surface: {pre_cp_source_path}"
    )

missing_source_columns = sorted(
    set(source_columns) - set(parquet_columns(pre_cp_source_path))
)

if missing_source_columns:
    raise ValueError(
        "Pre-CP baseline prediction surface is missing required columns: "
        f"{missing_source_columns}"
    )

pre_cp_source_df = pd.read_parquet(
    pre_cp_source_path,
    columns=source_columns,
    filters=[("stage", "==", "rolling_validation")],
)

pre_cp_source_df["target_date"] = pd.to_datetime(
    pre_cp_source_df["target_date"]
).dt.normalize()


# ---------------------------------------------------------------------
# 5) Preserve the legal Pre-CP scoring population
# ---------------------------------------------------------------------
# Native eligibility and target availability define whether a row belongs
# to the Pre-CP validation scoring problem. Benchmark availability is NOT
# used here because downstream code must be able to observe its absence.

native_eligible = (
    pre_cp_source_df["native_track_eligible"]
    .fillna(False)
    .astype(bool)
)

target_available = (
    pre_cp_source_df["target_label_available"]
    .fillna(False)
    .astype(bool)
)

within_validation_window = pre_cp_source_df["target_date"].between(
    validation_start,
    validation_end,
    inclusive="both",
)

selected_pre_cp_benchmark_df = pre_cp_source_df.loc[
    native_eligible
    & target_available
    & within_validation_window
].copy()

if selected_pre_cp_benchmark_df.empty:
    raise AssertionError(
        "No legal Pre-CP rolling-validation rows remain."
    )

if not selected_pre_cp_benchmark_df["modeling_track"].eq(
    "pre_cp_reference"
).all():
    raise AssertionError(
        "Pre-CP benchmark surface contains an unexpected modeling track."
    )

if not selected_pre_cp_benchmark_df["stage"].eq(
    "rolling_validation"
).all():
    raise AssertionError(
        "Pre-CP benchmark surface contains an unexpected evaluation stage."
    )

if selected_pre_cp_benchmark_df["target_observed_value"].isna().any():
    raise AssertionError(
        "A legal Pre-CP validation row is missing its observed target."
    )

if (
    selected_pre_cp_benchmark_df["target_date"].min() != validation_start
    or selected_pre_cp_benchmark_df["target_date"].max() != validation_end
):
    raise AssertionError(
        "Pre-CP benchmark surface does not span the frozen validation window."
    )


# ---------------------------------------------------------------------
# 6) Attach the selected benchmark and expose one stable prediction field
# ---------------------------------------------------------------------

selected_pre_cp_benchmark_df = selected_pre_cp_benchmark_df.merge(
    pre_cp_benchmark_registry_df,
    on=["modeling_track", "metric", "horizon"],
    how="left",
    validate="many_to_one",
)

if selected_pre_cp_benchmark_df["baseline_id"].isna().any():
    raise AssertionError(
        "At least one Pre-CP validation row lacks a benchmark assignment."
    )

selected_pre_cp_benchmark_df["benchmark_prediction"] = pd.Series(
    np.nan,
    index=selected_pre_cp_benchmark_df.index,
    dtype=float,
)

for baseline_id in selected_baseline_ids:
    prediction_column = f"baseline__{baseline_id}"
    benchmark_rows = selected_pre_cp_benchmark_df["baseline_id"].eq(
        baseline_id
    )

    selected_pre_cp_benchmark_df.loc[
        benchmark_rows,
        "benchmark_prediction",
    ] = pd.to_numeric(
        selected_pre_cp_benchmark_df.loc[
            benchmark_rows,
            prediction_column,
        ],
        errors="coerce",
    )

selected_pre_cp_benchmark_df["benchmark_prediction_available"] = (
    selected_pre_cp_benchmark_df["benchmark_prediction"].notna()
)


# ---------------------------------------------------------------------
# 7) Preserve the two benchmark-support definitions
# ---------------------------------------------------------------------

selected_pre_cp_benchmark_df[
    "baseline_tournament_support_eligible"
] = (
    selected_pre_cp_benchmark_df["native_track_eligible"]
    .fillna(False)
    .astype(bool)
    & selected_pre_cp_benchmark_df["target_label_available"]
        .fillna(False)
        .astype(bool)
    & selected_pre_cp_benchmark_df["all_baselines_available"]
        .fillna(False)
        .astype(bool)
)

selected_pre_cp_benchmark_df[
    "selected_benchmark_support_eligible"
] = (
    selected_pre_cp_benchmark_df["native_track_eligible"]
    .fillna(False)
    .astype(bool)
    & selected_pre_cp_benchmark_df["target_label_available"]
        .fillna(False)
        .astype(bool)
    & selected_pre_cp_benchmark_df["benchmark_prediction_available"]
)

support_subset_match = (
    ~selected_pre_cp_benchmark_df[
        "baseline_tournament_support_eligible"
    ]
    | selected_pre_cp_benchmark_df[
        "selected_benchmark_support_eligible"
    ]
).all()

if not support_subset_match:
    raise AssertionError(
        "Tournament support is not contained within selected-benchmark support."
    )


# ---------------------------------------------------------------------
# 8) Reduce to the explicit downstream contract
# ---------------------------------------------------------------------

selected_pre_cp_columns = [
    "modeling_track",
    "stage",
    "taxi_zone_id",
    "metric",
    "forecast_sequence_id",
    "origin_observation_sequence_id",
    "target_observation_sequence_id",
    "horizon",
    "target_date",
    "target_observed_value",
    "target_label_available",
    "native_track_eligible",
    "all_baselines_available",
    "baseline_id",
    "baseline_label",
    "selection_metric",
    "selection_stage",
    "benchmark_prediction",
    "benchmark_prediction_available",
    "baseline_tournament_support_eligible",
    "selected_benchmark_support_eligible",
]

selected_pre_cp_benchmark_df = (
    selected_pre_cp_benchmark_df[selected_pre_cp_columns]
    .sort_values(
        [
            "metric",
            "horizon",
            "forecast_sequence_id",
            "origin_observation_sequence_id",
            "target_observation_sequence_id",
        ]
    )
    .reset_index(drop=True)
)

forecast_key = [
    "modeling_track",
    "metric",
    "horizon",
    "forecast_sequence_id",
    "taxi_zone_id",
    "origin_observation_sequence_id",
    "target_observation_sequence_id",
]

duplicate_forecast_rows = int(
    selected_pre_cp_benchmark_df.duplicated(forecast_key).sum()
)

if duplicate_forecast_rows:
    raise AssertionError(
        "Selected Pre-CP benchmark surface contains duplicate forecast rows."
    )

job_count = int(
    selected_pre_cp_benchmark_df[
        ["metric", "horizon"]
    ].drop_duplicates().shape[0]
)

if job_count != 15:
    raise AssertionError(
        f"Expected 15 Pre-CP Target × Horizon jobs; found {job_count}."
    )


# ---------------------------------------------------------------------
# 9) Reproduce the frozen benchmark result on tournament support
# ---------------------------------------------------------------------
# Re-scoring the selected benchmark on the same common-support denominator
# must reproduce the benchmark contract exactly.

benchmark_scoring_df = selected_pre_cp_benchmark_df.loc[
    selected_pre_cp_benchmark_df[
        "baseline_tournament_support_eligible"
    ],
    [
        "metric",
        "horizon",
        "baseline_id",
        "target_observed_value",
        "benchmark_prediction",
    ],
].copy()

if benchmark_scoring_df["benchmark_prediction"].isna().any():
    raise AssertionError(
        "Tournament-support rows contain unavailable selected benchmarks."
    )

benchmark_scoring_df["absolute_error"] = (
    benchmark_scoring_df["benchmark_prediction"]
    - benchmark_scoring_df["target_observed_value"]
).abs()

published_score_df = (
    benchmark_scoring_df
    .groupby(
        ["metric", "horizon", "baseline_id"],
        observed=True,
    )
    .agg(
        published_evaluated_rows=("absolute_error", "size"),
        published_mae=("absolute_error", "mean"),
    )
    .reset_index()
)

frozen_pre_cp_score_df = (
    frozen_benchmark_contract_df.loc[
        frozen_benchmark_contract_df["experiment_id"].eq(
            "pre_cp_counterfactual"
        ),
        [
            "metric",
            "horizon",
            "baseline_id",
            "evaluated_rows",
            "mae",
        ],
    ]
    .copy()
)

score_reconciliation_df = frozen_pre_cp_score_df.merge(
    published_score_df,
    on=["metric", "horizon", "baseline_id"],
    how="left",
    validate="one_to_one",
)

score_support_match = (
    len(score_reconciliation_df) == 15
    and score_reconciliation_df[
        "published_evaluated_rows"
    ].eq(
        score_reconciliation_df["evaluated_rows"]
    ).all()
)

score_mae_match = (
    len(score_reconciliation_df) == 15
    and np.isclose(
        score_reconciliation_df["published_mae"],
        score_reconciliation_df["mae"],
        rtol=1e-12,
        atol=1e-12,
    ).all()
)

if not score_support_match:
    raise AssertionError(
        "Published Pre-CP benchmark support does not match the frozen contract."
    )

if not score_mae_match:
    raise AssertionError(
        "Published Pre-CP benchmark MAE does not match the frozen contract."
    )


# ---------------------------------------------------------------------
# 10) Persist the final row-level benchmark handoff
# ---------------------------------------------------------------------

if WRITE_FINAL_OUTPUTS:
    selected_pre_cp_benchmark_df.to_parquet(
        SELECTED_PRE_CP_BENCHMARK_PREDICTIONS_PATH,
        index=False,
        compression="zstd",
    )

persisted_metadata = parquet_metadata(
    SELECTED_PRE_CP_BENCHMARK_PREDICTIONS_PATH
)

if persisted_metadata["rows"] != len(selected_pre_cp_benchmark_df):
    raise AssertionError(
        "Persisted Pre-CP benchmark row count does not match memory."
    )

if persisted_metadata["columns"] != len(selected_pre_cp_columns):
    raise AssertionError(
        "Persisted Pre-CP benchmark column count does not match the contract."
    )


# ---------------------------------------------------------------------
# 11) Summarize the published benchmark surface
# ---------------------------------------------------------------------

selected_pre_cp_benchmark_summary_df = (
    selected_pre_cp_benchmark_df
    .groupby(
        ["metric", "horizon", "baseline_id", "baseline_label"],
        observed=True,
    )
    .agg(
        validation_rows=("metric", "size"),
        benchmark_available_rows=(
            "benchmark_prediction_available",
            "sum",
        ),
        tournament_support_rows=(
            "baseline_tournament_support_eligible",
            "sum",
        ),
        selected_benchmark_support_rows=(
            "selected_benchmark_support_eligible",
            "sum",
        ),
    )
    .reset_index()
    .assign(
        Target=lambda frame: frame["metric"].map(TARGET_LABELS)
    )
)

display(
    selected_pre_cp_benchmark_summary_df[
        [
            "Target",
            "horizon",
            "baseline_label",
            "validation_rows",
            "benchmark_available_rows",
            "tournament_support_rows",
            "selected_benchmark_support_rows",
        ]
    ].rename(columns={
        "horizon": "Horizon",
        "baseline_label": "Frozen benchmark",
        "validation_rows": "Legal validation rows",
        "benchmark_available_rows": "Benchmark available",
        "tournament_support_rows": "Tournament support",
        "selected_benchmark_support_rows": "Selected-benchmark support",
    })
)

print(
    "PASS — selected Pre-CP benchmark surface published.\n"
    f"Rows: {len(selected_pre_cp_benchmark_df):,}\n"
    f"Columns: {len(selected_pre_cp_columns)}\n"
    f"Jobs: {job_count}"
)

del (
    pre_cp_source_df,
    benchmark_scoring_df,
    published_score_df,
    frozen_pre_cp_score_df,
    score_reconciliation_df,
    track_contract_df,
    pre_cp_track_contract_df,
)

gc.collect()

,Target,Horizon,Frozen benchmark,Legal validation rows,Benchmark available,Tournament support,Selected-benchmark support
0,FHVHV average speed,1,Previous week · same Daypart,469258,468452,466534,468452
1,FHVHV average speed,2,Previous week · same Daypart,469258,468452,466353,468452
2,FHVHV average speed,5,Previous week · same Daypart,469258,468452,467224,468452
3,FHVHV trips,1,Previous week · same Daypart,471173,470428,468628,470428
4,FHVHV trips,2,Previous week · same Daypart,471173,470428,468467,470428
5,FHVHV trips,5,Previous week · same Daypart,471173,470428,469270,470428
6,Subway ridership,1,Previous week · same Daypart,284235,283371,281584,283371
7,Subway ridership,2,Previous week · same Daypart,284235,283371,281430,283371
8,Subway ridership,5,Previous week · same Daypart,284235,283371,282065,283371
9,Taxi average speed,1,Recent 7-day Daypart mean,197689,193236,187772,193236


PASS — selected Pre-CP benchmark surface published.
Rows: 5,680,584
Columns: 21
Jobs: 15


319

**Findings.** The selected Pre-CP benchmark is now available at row level across all **15 Target × Horizon jobs** in the legal 2024 validation population.

The handoff preserves rows where the selected benchmark is unavailable rather than silently dropping them. Genuine observed zeros remain valid scoring outcomes because target availability is tracked separately from the observed value itself.

`baseline_tournament_support_eligible` identifies the common-support population used to select among all four simple baselines, while `selected_benchmark_support_eligible` identifies the broader population available when only the frozen selected benchmark is required.

On the tournament-support population, the published benchmark predictions reproduce the frozen 4.2.4 MAE and evaluated-row counts exactly.

**4.2.4.6B — Run final baseline and forecastability QA.**

Validate the complete baseline package before handoff.

The audit confirms the candidate definitions, benchmark-selection results, forecasting scope, forecastability evidence, and the selected Pre-CP row-level benchmark surface.

The row-level checks verify that benchmark identities match the frozen registry, forecast rows are unique, benchmark availability is explicit, the two support flags preserve their defined populations, and the published benchmark predictions reproduce the frozen Pre-CP benchmark results on the tournament-support population.


In [21]:
# ---------------------------------------------------------------------
# 4.2.4.6B — Run final baseline and forecastability QA
# ---------------------------------------------------------------------

# ---------------------------------------------------------------------
# 1) Load the compact final contracts
# ---------------------------------------------------------------------
# Section 6 can validate the completed handoff directly from persisted
# outputs without repeating baseline prediction construction or selection.

baseline_sequence_performance_path = (
    FINAL_DIR / "baseline_sequence_performance.parquet"
)

frozen_benchmark_contract_path = (
    FINAL_DIR / "frozen_benchmark_contract.parquet"
)

supported_modeling_scope_path = (
    FINAL_DIR / "supported_modeling_scope.parquet"
)

baseline_definition_registry_df = pd.read_parquet(
    BASELINE_DEFINITION_PATH
)

baseline_leaderboard_df = pd.read_parquet(
    BASELINE_LEADERBOARD_PATH
)

baseline_sequence_performance_df = pd.read_parquet(
    baseline_sequence_performance_path
)

frozen_benchmark_contract_df = pd.read_parquet(
    frozen_benchmark_contract_path
)

advanced_model_benchmark_registry_df = pd.read_parquet(
    FROZEN_BENCHMARK_PATH
)

empirical_forecastability_df = pd.read_parquet(
    FORECASTABILITY_SUMMARY_PATH
)

supported_modeling_scope_df = pd.read_parquet(
    supported_modeling_scope_path
)

selected_pre_cp_qa_df = pd.read_parquet(
    SELECTED_PRE_CP_BENCHMARK_PREDICTIONS_PATH
)

selected_pre_cp_qa_df["target_date"] = pd.to_datetime(
    selected_pre_cp_qa_df["target_date"]
).dt.normalize()


# ---------------------------------------------------------------------
# 2) Reconstruct the minimum validation MAE for every benchmark slot
# ---------------------------------------------------------------------

minimum_mae_qa_df = (
    baseline_leaderboard_df
    .groupby(
        ["experiment_id", "metric", "horizon"],
        observed=True,
    )["mae"]
    .min()
    .rename("minimum_validation_mae")
    .reset_index()
)

frozen_minimum_qa_df = frozen_benchmark_contract_df.merge(
    minimum_mae_qa_df,
    on=["experiment_id", "metric", "horizon"],
    how="left",
    validate="one_to_one",
)

winner_matches_minimum = np.isclose(
    frozen_minimum_qa_df["mae"],
    frozen_minimum_qa_df["minimum_validation_mae"],
    rtol=1e-12,
    atol=1e-12,
).all()


# ---------------------------------------------------------------------
# 3) Verify that Post-CP and Full-History inherit the same benchmark
# ---------------------------------------------------------------------

post_full_qa_df = (
    advanced_model_benchmark_registry_df.loc[
        advanced_model_benchmark_registry_df["modeling_track"].isin(
            ["post_cp_primary", "full_history_regime_aware"]
        ),
        ["modeling_track", "metric", "horizon", "baseline_id"],
    ]
    .pivot(
        index=["metric", "horizon"],
        columns="modeling_track",
        values="baseline_id",
    )
)

post_full_same_benchmark = (
    post_full_qa_df.shape == (15, 2)
    and post_full_qa_df["post_cp_primary"].eq(
        post_full_qa_df["full_history_regime_aware"]
    ).all()
)


# ---------------------------------------------------------------------
# 4) Validate the selected Pre-CP row-level handoff
# ---------------------------------------------------------------------

selected_pre_cp_job_count = int(
    selected_pre_cp_qa_df[
        ["metric", "horizon"]
    ].drop_duplicates().shape[0]
)

pre_cp_track_contract_df = (
    pd.read_parquet(
        TRACK_CONTRACT_PATH,
        columns=[
            "modeling_track",
            "rolling_validation_target_start_date",
            "rolling_validation_target_end_date",
        ],
    )
    .loc[
        lambda frame: frame["modeling_track"].eq(
            "pre_cp_reference"
        )
    ]
    .copy()
)

if len(pre_cp_track_contract_df) != 1:
    raise AssertionError(
        "Expected exactly one Pre-CP Reference track-contract row."
    )

expected_validation_start = pd.to_datetime(
    pre_cp_track_contract_df.iloc[0][
        "rolling_validation_target_start_date"
    ]
).normalize()

expected_validation_end = pd.to_datetime(
    pre_cp_track_contract_df.iloc[0][
        "rolling_validation_target_end_date"
    ]
).normalize()

selected_pre_cp_scope_match = (
    selected_pre_cp_job_count == 15
    and selected_pre_cp_qa_df["modeling_track"]
        .eq("pre_cp_reference")
        .all()
    and selected_pre_cp_qa_df["stage"]
        .eq("rolling_validation")
        .all()
    and selected_pre_cp_qa_df["native_track_eligible"]
        .fillna(False)
        .astype(bool)
        .all()
    and selected_pre_cp_qa_df["target_label_available"]
        .fillna(False)
        .astype(bool)
        .all()
    and selected_pre_cp_qa_df["target_date"].min()
        == expected_validation_start
    and selected_pre_cp_qa_df["target_date"].max()
        == expected_validation_end
)

selected_pre_cp_forecast_key = [
    "modeling_track",
    "metric",
    "horizon",
    "forecast_sequence_id",
    "taxi_zone_id",
    "origin_observation_sequence_id",
    "target_observation_sequence_id",
]

selected_pre_cp_duplicate_rows = int(
    selected_pre_cp_qa_df.duplicated(
        selected_pre_cp_forecast_key
    ).sum()
)

expected_pre_cp_registry_df = (
    advanced_model_benchmark_registry_df.loc[
        advanced_model_benchmark_registry_df[
            "modeling_track"
        ].eq("pre_cp_reference"),
        [
            "modeling_track",
            "metric",
            "horizon",
            "baseline_id",
            "baseline_label",
            "selection_metric",
            "selection_stage",
        ],
    ]
    .drop_duplicates()
    .sort_values(["modeling_track", "metric", "horizon"])
    .reset_index(drop=True)
)

observed_pre_cp_registry_df = (
    selected_pre_cp_qa_df[
        [
            "modeling_track",
            "metric",
            "horizon",
            "baseline_id",
            "baseline_label",
            "selection_metric",
            "selection_stage",
        ]
    ]
    .drop_duplicates()
    .sort_values(["modeling_track", "metric", "horizon"])
    .reset_index(drop=True)
)

selected_pre_cp_registry_match = (
    observed_pre_cp_registry_df.equals(
        expected_pre_cp_registry_df
    )
)

benchmark_availability_match = (
    selected_pre_cp_qa_df[
        "benchmark_prediction_available"
    ]
    .fillna(False)
    .astype(bool)
    .eq(
        selected_pre_cp_qa_df[
            "benchmark_prediction"
        ].notna()
    )
    .all()
)

target_availability_match = (
    selected_pre_cp_qa_df[
        "target_label_available"
    ]
    .fillna(False)
    .astype(bool)
    .eq(
        selected_pre_cp_qa_df[
            "target_observed_value"
        ].notna()
    )
    .all()
)

expected_tournament_support = (
    selected_pre_cp_qa_df["native_track_eligible"]
    .fillna(False)
    .astype(bool)
    & selected_pre_cp_qa_df["target_label_available"]
        .fillna(False)
        .astype(bool)
    & selected_pre_cp_qa_df["all_baselines_available"]
        .fillna(False)
        .astype(bool)
)

expected_selected_support = (
    selected_pre_cp_qa_df["native_track_eligible"]
    .fillna(False)
    .astype(bool)
    & selected_pre_cp_qa_df["target_label_available"]
        .fillna(False)
        .astype(bool)
    & selected_pre_cp_qa_df["benchmark_prediction"]
        .notna()
)

support_flags_match = (
    selected_pre_cp_qa_df[
        "baseline_tournament_support_eligible"
    ]
    .fillna(False)
    .astype(bool)
    .eq(expected_tournament_support)
    .all()
    and selected_pre_cp_qa_df[
        "selected_benchmark_support_eligible"
    ]
    .fillna(False)
    .astype(bool)
    .eq(expected_selected_support)
    .all()
)

support_subset_match = (
    ~selected_pre_cp_qa_df[
        "baseline_tournament_support_eligible"
    ].fillna(False).astype(bool)
    | selected_pre_cp_qa_df[
        "selected_benchmark_support_eligible"
    ].fillna(False).astype(bool)
).all()


# ---------------------------------------------------------------------
# 5) Reconcile the row-level benchmark with the frozen Pre-CP contract
# ---------------------------------------------------------------------

selected_pre_cp_score_df = selected_pre_cp_qa_df.loc[
    selected_pre_cp_qa_df[
        "baseline_tournament_support_eligible"
    ].fillna(False).astype(bool),
    [
        "metric",
        "horizon",
        "baseline_id",
        "target_observed_value",
        "benchmark_prediction",
    ],
].copy()

selected_pre_cp_score_df["absolute_error"] = (
    selected_pre_cp_score_df["benchmark_prediction"]
    - selected_pre_cp_score_df["target_observed_value"]
).abs()

published_pre_cp_score_df = (
    selected_pre_cp_score_df
    .groupby(
        ["metric", "horizon", "baseline_id"],
        observed=True,
    )
    .agg(
        published_evaluated_rows=("absolute_error", "size"),
        published_mae=("absolute_error", "mean"),
    )
    .reset_index()
)

frozen_pre_cp_score_df = (
    frozen_benchmark_contract_df.loc[
        frozen_benchmark_contract_df["experiment_id"].eq(
            "pre_cp_counterfactual"
        ),
        [
            "metric",
            "horizon",
            "baseline_id",
            "evaluated_rows",
            "mae",
        ],
    ]
    .copy()
)

pre_cp_score_reconciliation_df = frozen_pre_cp_score_df.merge(
    published_pre_cp_score_df,
    on=["metric", "horizon", "baseline_id"],
    how="left",
    validate="one_to_one",
)

selected_pre_cp_score_match = (
    len(pre_cp_score_reconciliation_df) == 15
    and pre_cp_score_reconciliation_df[
        "published_evaluated_rows"
    ].eq(
        pre_cp_score_reconciliation_df["evaluated_rows"]
    ).all()
    and np.isclose(
        pre_cp_score_reconciliation_df["published_mae"],
        pre_cp_score_reconciliation_df["mae"],
        rtol=1e-12,
        atol=1e-12,
    ).all()
)


# ---------------------------------------------------------------------
# 6) Precompute compact QA details
# ---------------------------------------------------------------------

matching_benchmark_slots = int(
    np.isclose(
        frozen_minimum_qa_df["mae"],
        frozen_minimum_qa_df["minimum_validation_mae"],
        rtol=1e-12,
        atol=1e-12,
    ).sum()
)

minimum_skill_pct = float(
    empirical_forecastability_df[
        "skill_vs_persistence_pct"
    ].min()
)

minimum_breadth_pct = float(
    empirical_forecastability_df[
        "share_beating_persistence_pct"
    ].min()
)

maximum_breadth_pct = float(
    empirical_forecastability_df[
        "share_beating_persistence_pct"
    ].max()
)

supported_target_count = int(
    supported_modeling_scope_df["metric"].nunique()
)

supported_horizons = sorted(
    int(value)
    for value in supported_modeling_scope_df["horizon"].unique()
)

supported_model_family_count = int(
    supported_modeling_scope_df["model_family"].nunique()
)

selection_stage_values = sorted(
    frozen_benchmark_contract_df["selection_stage"]
    .dropna()
    .astype(str)
    .unique()
)


# ---------------------------------------------------------------------
# 7) Assemble the final QA checks
# ---------------------------------------------------------------------

final_qa_rows = [
    qa_row(
        "Four candidate baseline rules are present",
        (
            len(baseline_definition_registry_df) == 4
            and baseline_definition_registry_df[
                "baseline_id"
            ].nunique() == 4
        ),
        (
            f"rows = {len(baseline_definition_registry_df):,}; "
            f"unique rules = "
            f"{baseline_definition_registry_df['baseline_id'].nunique():,}"
        ),
    ),

    qa_row(
        "Candidate leaderboard contains every experiment × target × horizon × rule",
        (
            len(baseline_leaderboard_df) == 120
            and not baseline_leaderboard_df.duplicated(
                [
                    "experiment_id",
                    "metric",
                    "horizon",
                    "baseline_id",
                ]
            ).any()
        ),
        f"rows = {len(baseline_leaderboard_df):,}; expected = 120",
    ),

    qa_row(
        "Frozen benchmark contract contains exactly 30 benchmark slots",
        (
            len(frozen_benchmark_contract_df) == 30
            and not frozen_benchmark_contract_df.duplicated(
                ["experiment_id", "metric", "horizon"]
            ).any()
        ),
        f"rows = {len(frozen_benchmark_contract_df):,}; expected = 30",
    ),

    qa_row(
        "Every frozen benchmark was selected from rolling validation",
        frozen_benchmark_contract_df[
            "selection_stage"
        ].eq("rolling_validation").all(),
        "selection_stage values = " + ", ".join(selection_stage_values),
    ),

    qa_row(
        "Every frozen benchmark matches the minimum validation MAE",
        winner_matches_minimum,
        f"matching benchmark slots = {matching_benchmark_slots}/30",
    ),

    qa_row(
        "Advanced-model registry contains one benchmark per Track × Target × Horizon",
        (
            len(advanced_model_benchmark_registry_df) == 45
            and not advanced_model_benchmark_registry_df.duplicated(
                ["modeling_track", "metric", "horizon"]
            ).any()
        ),
        (
            f"rows = {len(advanced_model_benchmark_registry_df):,}; "
            "expected = 45"
        ),
    ),

    qa_row(
        "Post-CP Primary and Full-History share the same benchmark",
        post_full_same_benchmark,
        (
            f"shared Target × Horizon slots checked = "
            f"{len(post_full_qa_df):,}"
        ),
    ),

    qa_row(
        "Forecastability summary contains every benchmark × target × horizon",
        (
            len(empirical_forecastability_df) == 30
            and not empirical_forecastability_df.duplicated(
                ["experiment_id", "metric", "horizon"]
            ).any()
        ),
        f"rows = {len(empirical_forecastability_df):,}; expected = 30",
    ),

    qa_row(
        "Every frozen benchmark has positive validation skill versus persistence",
        empirical_forecastability_df[
            "skill_vs_persistence_pct"
        ].gt(0).all(),
        f"minimum skill = {minimum_skill_pct:.3f}%",
    ),

    qa_row(
        "Sequence-level breadth percentages remain valid",
        empirical_forecastability_df[
            "share_beating_persistence_pct"
        ].between(0, 100, inclusive="both").all(),
        (
            f"range = {minimum_breadth_pct:.3f}% "
            f"to {maximum_breadth_pct:.3f}%"
        ),
    ),

    qa_row(
        "Supported modeling scope contains all 135 downstream assignments",
        (
            len(supported_modeling_scope_df) == 135
            and not supported_modeling_scope_df.duplicated(
                [
                    "model_family",
                    "modeling_track",
                    "metric",
                    "horizon",
                ]
            ).any()
        ),
        (
            f"rows = {len(supported_modeling_scope_df):,}; "
            "expected = 135"
        ),
    ),

    qa_row(
        "All five targets and three horizons continue into every model family",
        (
            supported_modeling_scope_df["scope_status"].eq("continue").all()
            and supported_target_count == 5
            and set(supported_horizons) == set(FORECAST_HORIZONS)
            and supported_model_family_count == 3
        ),
        (
            f"targets = {supported_target_count}; "
            f"horizons = {supported_horizons}; "
            f"model families = {supported_model_family_count}"
        ),
    ),

    qa_row(
        "Selected Pre-CP benchmark surface has the frozen validation scope",
        selected_pre_cp_scope_match,
        (
            f"jobs = {selected_pre_cp_job_count}; "
            f"window = {expected_validation_start.date()} "
            f"to {expected_validation_end.date()}"
        ),
    ),

    qa_row(
        "Selected Pre-CP benchmark surface has unique forecast rows",
        selected_pre_cp_duplicate_rows == 0,
        f"duplicate forecast keys = {selected_pre_cp_duplicate_rows:,}",
    ),

    qa_row(
        "Selected Pre-CP benchmark identity matches the frozen registry",
        selected_pre_cp_registry_match,
        (
            f"observed jobs = {len(observed_pre_cp_registry_df):,}; "
            f"expected jobs = {len(expected_pre_cp_registry_df):,}"
        ),
    ),

    qa_row(
        "Selected Pre-CP target and benchmark availability flags are exact",
        target_availability_match and benchmark_availability_match,
        (
            f"target flag match = {target_availability_match}; "
            f"benchmark flag match = {benchmark_availability_match}"
        ),
    ),

    qa_row(
        "Selected Pre-CP support flags preserve their definitions",
        support_flags_match and support_subset_match,
        (
            "tournament = native + observed + all four available; "
            "selected = native + observed + selected benchmark available"
        ),
    ),

    qa_row(
        "Selected Pre-CP benchmark reproduces frozen MAE and evaluated-row support",
        selected_pre_cp_score_match,
        (
            f"reconciled Target × Horizon jobs = "
            f"{len(pre_cp_score_reconciliation_df):,}/15"
        ),
    ),
]

if len(final_qa_rows) != 18:
    raise AssertionError(
        f"Expected 18 final QA checks; found {len(final_qa_rows)}."
    )


# ---------------------------------------------------------------------
# 8) Persist and display the final QA result
# ---------------------------------------------------------------------

final_baseline_qa_df = pd.DataFrame(final_qa_rows)

if WRITE_FINAL_OUTPUTS:
    final_baseline_qa_df.to_parquet(
        FINAL_QA_424_PATH,
        index=False,
        compression="zstd",
    )

failed_qa_df = final_baseline_qa_df.loc[
    ~final_baseline_qa_df["status"].eq("PASS")
]

if not failed_qa_df.empty:
    raise AssertionError(
        "4.2.4 final QA failed:\n"
        + failed_qa_df[
            ["check", "detail"]
        ].to_string(index=False)
    )

display(final_baseline_qa_df)

del (
    selected_pre_cp_qa_df,
    selected_pre_cp_score_df,
    published_pre_cp_score_df,
    frozen_pre_cp_score_df,
    pre_cp_score_reconciliation_df,
    pre_cp_track_contract_df,
)

gc.collect()

,check,status,detail
0,Four candidate baseline rules are present,PASS,rows = 4; unique rules = 4
1,Candidate leaderboard contains every experimen...,PASS,rows = 120; expected = 120
2,Frozen benchmark contract contains exactly 30 ...,PASS,rows = 30; expected = 30
3,Every frozen benchmark was selected from rolli...,PASS,selection_stage values = rolling_validation
4,Every frozen benchmark matches the minimum val...,PASS,matching benchmark slots = 30/30
5,Advanced-model registry contains one benchmark...,PASS,rows = 45; expected = 45
6,Post-CP Primary and Full-History share the sam...,PASS,shared Target × Horizon slots checked = 15
7,Forecastability summary contains every benchma...,PASS,rows = 30; expected = 30
8,Every frozen benchmark has positive validation...,PASS,minimum skill = 15.112%
9,Sequence-level breadth percentages remain valid,PASS,range = 79.457% to 100.000%


0

**Findings.** The completed baseline package passes **all 18 final QA checks**.

The original benchmark contract remains complete and internally consistent: all 120 candidate-performance rows are present, every one of the 30 frozen benchmark slots matches the minimum validation MAE, all 45 Modeling Track × Target × Horizon benchmark assignments are present, and the full 135-job downstream modeling scope remains intact.

The selected Pre-CP row-level handoff also passes every downstream-contract check. All 15 Target × Horizon jobs are present, forecast rows are unique, benchmark identities match the frozen registry, target and benchmark availability are represented explicitly, and the two support flags preserve their defined populations.

Finally, the row-level selected benchmark reproduces the frozen Pre-CP benchmark MAE and evaluated-row counts exactly on the tournament-support population.

**4.2.4.6C — Package the final baseline handoff.**

The final baseline package provides the contracts and evidence required by later forecasting notebooks without requiring them to reconstruct baseline logic.

It includes:

* the four baseline definitions;
* validation performance for every candidate;
* Forecast Sequence-level performance evidence;
* the frozen benchmark contracts;
* the benchmark assigned to each modeling track;
* the selected Pre-CP benchmark prediction at row level;
* the forecastability summary;
* the supported downstream modeling scope; and
* the final QA results.

The manifest records each authoritative output together with its location, dimensions, and downstream role.

In [22]:
# ---------------------------------------------------------------------
# 4.2.4.6C — Build the final baseline handoff manifest
# ---------------------------------------------------------------------

# ---------------------------------------------------------------------
# 1) Define the final downstream assets
# ---------------------------------------------------------------------
# Compact tables are already in memory from final QA. The row-level Pre-CP
# benchmark is represented through Parquet metadata so the manifest does
# not reload a multi-row forecast surface solely to inventory it.

selected_pre_cp_metadata = parquet_metadata(
    SELECTED_PRE_CP_BENCHMARK_PREDICTIONS_PATH
)

if selected_pre_cp_metadata["columns"] != 21:
    raise AssertionError(
        "Selected Pre-CP benchmark handoff does not match its 21-column contract."
    )

final_asset_specs = [
    (
        "baseline_definition_registry",
        BASELINE_DEFINITION_PATH,
        baseline_definition_registry_df,
        None,
        None,
        "Defines the four simple candidate forecasting rules.",
    ),
    (
        "baseline_leaderboard",
        BASELINE_LEADERBOARD_PATH,
        baseline_leaderboard_df,
        None,
        None,
        "Stores validation performance for every candidate rule.",
    ),
    (
        "baseline_sequence_performance",
        baseline_sequence_performance_path,
        baseline_sequence_performance_df,
        None,
        None,
        "Stores Forecast Sequence-level candidate performance.",
    ),
    (
        "frozen_benchmark_contract",
        frozen_benchmark_contract_path,
        frozen_benchmark_contract_df,
        None,
        None,
        "Stores the validation-selected benchmark for each benchmark problem.",
    ),
    (
        "advanced_model_benchmark_registry",
        FROZEN_BENCHMARK_PATH,
        advanced_model_benchmark_registry_df,
        None,
        None,
        "Maps each frozen benchmark into its downstream modeling track.",
    ),
    (
        "selected_benchmark_predictions__pre_cp_reference",
        SELECTED_PRE_CP_BENCHMARK_PREDICTIONS_PATH,
        None,
        selected_pre_cp_metadata["rows"],
        21,
        (
            "Provides the selected Pre-CP benchmark prediction and support "
            "context for each legal 2024 rolling-validation forecast row."
        ),
    ),
    (
        "empirical_forecastability_summary",
        FORECASTABILITY_SUMMARY_PATH,
        empirical_forecastability_df,
        None,
        None,
        "Summarizes benchmark error, skill, breadth, and horizon stability.",
    ),
    (
        "supported_modeling_scope",
        supported_modeling_scope_path,
        supported_modeling_scope_df,
        None,
        None,
        "Carries the approved forecasting jobs into downstream model families.",
    ),
    (
        "final_baseline_forecastability_qa",
        FINAL_QA_424_PATH,
        final_baseline_qa_df,
        None,
        None,
        "Records the final integrity checks for the 4.2.4 handoff.",
    ),
]


# ---------------------------------------------------------------------
# 2) Verify persisted dimensions and assemble the manifest
# ---------------------------------------------------------------------

manifest_records = []

for (
    asset,
    path,
    source_df,
    expected_rows,
    expected_columns,
    role,
) in final_asset_specs:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing final 4.2.4 asset: {path}"
        )

    metadata = parquet_metadata(path)

    if source_df is not None:
        expected_rows = len(source_df)
        expected_columns = source_df.shape[1]

    if metadata["rows"] != expected_rows:
        raise AssertionError(
            f"{asset}: persisted rows {metadata['rows']:,} "
            f"!= expected rows {expected_rows:,}."
        )

    if metadata["columns"] != expected_columns:
        raise AssertionError(
            f"{asset}: persisted columns {metadata['columns']:,} "
            f"!= expected columns {expected_columns:,}."
        )

    manifest_records.append({
        "asset": asset,
        "path": str(path),
        "rows": metadata["rows"],
        "columns": metadata["columns"],
        "size_mb": metadata["size_gb"] * 1024,
        "role": role,
    })

baseline_forecastability_manifest_df = pd.DataFrame(
    manifest_records
)

if len(baseline_forecastability_manifest_df) != 9:
    raise AssertionError(
        "Expected 9 authoritative assets in the 4.2.4 manifest."
    )

if WRITE_FINAL_OUTPUTS:
    baseline_forecastability_manifest_df.to_parquet(
        FINAL_MANIFEST_424_PATH,
        index=False,
        compression="zstd",
    )


# ---------------------------------------------------------------------
# 3) Show the final production manifest
# ---------------------------------------------------------------------

manifest_display_df = baseline_forecastability_manifest_df.copy()
manifest_display_df["size_mb"] = (
    manifest_display_df["size_mb"].round(3)
)

manifest_display_df.columns = [
    "Asset",
    "Path",
    "Rows",
    "Columns",
    "Size MB",
    "Downstream role",
]

display(manifest_display_df)# ---------------------------------------------------------------------
# 4.2.4.6C — Build the final baseline handoff manifest
# ---------------------------------------------------------------------

# ---------------------------------------------------------------------
# 1) Define the final downstream assets
# ---------------------------------------------------------------------
# Compact tables are already in memory from final QA. The row-level Pre-CP
# benchmark is represented through Parquet metadata so the manifest does
# not reload a multi-row forecast surface solely to inventory it.

selected_pre_cp_metadata = parquet_metadata(
    SELECTED_PRE_CP_BENCHMARK_PREDICTIONS_PATH
)

if selected_pre_cp_metadata["columns"] != 21:
    raise AssertionError(
        "Selected Pre-CP benchmark handoff does not match its 21-column contract."
    )

final_asset_specs = [
    (
        "baseline_definition_registry",
        BASELINE_DEFINITION_PATH,
        baseline_definition_registry_df,
        None,
        None,
        "Defines the four simple candidate forecasting rules.",
    ),
    (
        "baseline_leaderboard",
        BASELINE_LEADERBOARD_PATH,
        baseline_leaderboard_df,
        None,
        None,
        "Stores validation performance for every candidate rule.",
    ),
    (
        "baseline_sequence_performance",
        baseline_sequence_performance_path,
        baseline_sequence_performance_df,
        None,
        None,
        "Stores Forecast Sequence-level candidate performance.",
    ),
    (
        "frozen_benchmark_contract",
        frozen_benchmark_contract_path,
        frozen_benchmark_contract_df,
        None,
        None,
        "Stores the validation-selected benchmark for each benchmark problem.",
    ),
    (
        "advanced_model_benchmark_registry",
        FROZEN_BENCHMARK_PATH,
        advanced_model_benchmark_registry_df,
        None,
        None,
        "Maps each frozen benchmark into its downstream modeling track.",
    ),
    (
        "selected_benchmark_predictions__pre_cp_reference",
        SELECTED_PRE_CP_BENCHMARK_PREDICTIONS_PATH,
        None,
        selected_pre_cp_metadata["rows"],
        21,
        (
            "Provides the selected Pre-CP benchmark prediction and support "
            "context for each legal 2024 rolling-validation forecast row."
        ),
    ),
    (
        "empirical_forecastability_summary",
        FORECASTABILITY_SUMMARY_PATH,
        empirical_forecastability_df,
        None,
        None,
        "Summarizes benchmark error, skill, breadth, and horizon stability.",
    ),
    (
        "supported_modeling_scope",
        supported_modeling_scope_path,
        supported_modeling_scope_df,
        None,
        None,
        "Carries the approved forecasting jobs into downstream model families.",
    ),
    (
        "final_baseline_forecastability_qa",
        FINAL_QA_424_PATH,
        final_baseline_qa_df,
        None,
        None,
        "Records the final integrity checks for the 4.2.4 handoff.",
    ),
]


# ---------------------------------------------------------------------
# 2) Verify persisted dimensions and assemble the manifest
# ---------------------------------------------------------------------

manifest_records = []

for (
    asset,
    path,
    source_df,
    expected_rows,
    expected_columns,
    role,
) in final_asset_specs:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing final 4.2.4 asset: {path}"
        )

    metadata = parquet_metadata(path)

    if source_df is not None:
        expected_rows = len(source_df)
        expected_columns = source_df.shape[1]

    if metadata["rows"] != expected_rows:
        raise AssertionError(
            f"{asset}: persisted rows {metadata['rows']:,} "
            f"!= expected rows {expected_rows:,}."
        )

    if metadata["columns"] != expected_columns:
        raise AssertionError(
            f"{asset}: persisted columns {metadata['columns']:,} "
            f"!= expected columns {expected_columns:,}."
        )

    manifest_records.append({
        "asset": asset,
        "path": str(path),
        "rows": metadata["rows"],
        "columns": metadata["columns"],
        "size_mb": metadata["size_gb"] * 1024,
        "role": role,
    })

baseline_forecastability_manifest_df = pd.DataFrame(
    manifest_records
)

if len(baseline_forecastability_manifest_df) != 9:
    raise AssertionError(
        "Expected 9 authoritative assets in the 4.2.4 manifest."
    )

if WRITE_FINAL_OUTPUTS:
    baseline_forecastability_manifest_df.to_parquet(
        FINAL_MANIFEST_424_PATH,
        index=False,
        compression="zstd",
    )


# ---------------------------------------------------------------------
# 3) Show the final production manifest
# ---------------------------------------------------------------------

manifest_display_df = baseline_forecastability_manifest_df.copy()
manifest_display_df["size_mb"] = (
    manifest_display_df["size_mb"].round(3)
)

manifest_display_df.columns = [
    "Asset",
    "Path",
    "Rows",
    "Columns",
    "Size MB",
    "Downstream role",
]

display(manifest_display_df)

,Asset,Path,Rows,Columns,Size MB,Downstream role
0,baseline_definition_registry,pipeline_data/4.2.4.final_tables/baseline_defi...,4,8,0.007,Defines the four simple candidate forecasting ...
1,baseline_leaderboard,pipeline_data/4.2.4.final_tables/baseline_lead...,120,15,0.017,Stores validation performance for every candid...
2,baseline_sequence_performance,pipeline_data/4.2.4.final_tables/baseline_sequ...,24972,11,0.230,Stores Forecast Sequence-level candidate perfo...
3,frozen_benchmark_contract,pipeline_data/4.2.4.final_tables/frozen_benchm...,30,18,0.016,Stores the validation-selected benchmark for e...
4,advanced_model_benchmark_registry,pipeline_data/4.2.4.final_tables/advanced_mode...,45,20,0.017,Maps each frozen benchmark into its downstream...
5,selected_benchmark_predictions__pre_cp_reference,pipeline_data/4.2.4.final_tables/selected_benc...,5680584,21,42.082,Provides the selected Pre-CP benchmark predict...
6,empirical_forecastability_summary,pipeline_data/4.2.4.final_tables/empirical_for...,30,19,0.016,"Summarizes benchmark error, skill, breadth, an..."
7,supported_modeling_scope,pipeline_data/4.2.4.final_tables/supported_mod...,135,16,0.014,Carries the approved forecasting jobs into dow...
8,final_baseline_forecastability_qa,pipeline_data/4.2.4.final_tables/final_baselin...,18,3,0.004,Records the final integrity checks for the 4.2...


,Asset,Path,Rows,Columns,Size MB,Downstream role
0,baseline_definition_registry,pipeline_data/4.2.4.final_tables/baseline_defi...,4,8,0.007,Defines the four simple candidate forecasting ...
1,baseline_leaderboard,pipeline_data/4.2.4.final_tables/baseline_lead...,120,15,0.017,Stores validation performance for every candid...
2,baseline_sequence_performance,pipeline_data/4.2.4.final_tables/baseline_sequ...,24972,11,0.230,Stores Forecast Sequence-level candidate perfo...
3,frozen_benchmark_contract,pipeline_data/4.2.4.final_tables/frozen_benchm...,30,18,0.016,Stores the validation-selected benchmark for e...
4,advanced_model_benchmark_registry,pipeline_data/4.2.4.final_tables/advanced_mode...,45,20,0.017,Maps each frozen benchmark into its downstream...
5,selected_benchmark_predictions__pre_cp_reference,pipeline_data/4.2.4.final_tables/selected_benc...,5680584,21,42.082,Provides the selected Pre-CP benchmark predict...
6,empirical_forecastability_summary,pipeline_data/4.2.4.final_tables/empirical_for...,30,19,0.016,"Summarizes benchmark error, skill, breadth, an..."
7,supported_modeling_scope,pipeline_data/4.2.4.final_tables/supported_mod...,135,16,0.014,Carries the approved forecasting jobs into dow...
8,final_baseline_forecastability_qa,pipeline_data/4.2.4.final_tables/final_baselin...,18,3,0.004,Records the final integrity checks for the 4.2...


**Findings.** The final 4.2.4 manifest enumerates **9 authoritative downstream assets**.

`advanced_model_benchmark_registry.parquet` defines the benchmark assigned to every Modeling Track × Target × Horizon, while `selected_benchmark_predictions__pre_cp_reference.parquet` provides the corresponding row-level Pre-CP validation predictions and support context required by Chapter 5.

The row-level candidate surfaces remain detailed evaluation assets, while downstream forecasting notebooks can consume the selected benchmark directly from the final handoff.

### Section Summary

The baseline package is complete for both ordinary forecasting and the Chapter 5 counterfactual workflow.

The final handoff preserves the candidate definitions, validation leaderboard, Forecast Sequence-level evidence, frozen benchmark contracts, forecastability summary, supported modeling scope, and final QA.

For the Pre-CP Reference track, the frozen benchmark is also available at row level across the legal 2024 rolling-validation population. Downstream models can therefore be evaluated against the same benchmark on aligned forecast rows while preserving both the common-support denominator used for benchmark selection and the selected benchmark's own availability.

## Notebook Close

Notebook 4.2.4 establishes the simple forecasting benchmark and the forecastability floor for the modeling chapter.

The downstream contract fixes five elements:

* all five forecasting targets continue at h=1, h=2, and h=5;
* the Post-CP forecasting benchmark is shared by Post-CP Primary and Full-History Regime-Aware;
* the Pre-CP Reference benchmark is defined separately for the counterfactual workflow;
* the selected Pre-CP benchmark predictions are available at row level for the legal 2024 rolling-validation population; and
* advanced forecasting families are evaluated against these frozen benchmarks rather than redefining them.

The next modeling stages can therefore focus on whether richer forecasting systems improve on the recurring historical patterns captured by these simple benchmarks.

### Final File Inventory for 4.2.4

Notebook 4.2.4 produces a small set of **final handoff files** for downstream forecasting, plus several **intermediate working files** retained for reproducibility and audit. The final files should be treated as the authoritative outputs of this notebook; the intermediate files do not need to be loaded by later modeling notebooks unless their underlying baseline calculations need to be inspected.

#### Final handoff files

- **`baseline_definition_registry.parquet`** — Defines the four deterministic candidate baseline rules used in the validation tournament: persistence, previous day · same Daypart, previous week · same Daypart, and the recent 7-day same-Daypart mean. This is the authoritative definition of what each baseline forecast represents.

- **`baseline_leaderboard.parquet`** — Stores validation performance for all four candidate rules across each Benchmark × Target × Horizon combination. It contains the evidence used to select the frozen benchmark, with **MAE as the sole winner-selection metric** and supporting error and coverage diagnostics retained for interpretation.

- **`baseline_sequence_performance.parquet`** — Stores baseline performance at the individual Forecast Sequence level. This supports the breadth analysis showing whether a rule's improvement over persistence occurs broadly across zones and series rather than being driven by a small number of unusually easy cases.

- **`frozen_benchmark_contract.parquet`** — Records the single validation-selected benchmark for every benchmark-selection Target × Horizon. It freezes the **Post-CP forecasting benchmark** and the separately selected **Pre-CP counterfactual benchmark** so later models cannot redefine their comparison after seeing downstream results.

- **`advanced_model_benchmark_registry.parquet`** — Maps the frozen benchmark contracts onto the three modeling tracks. Post-CP Primary and Full-History Regime-Aware inherit the same Post-CP forecasting benchmark, while Pre-CP Reference inherits the separately selected counterfactual benchmark.

- **`selected_benchmark_predictions__pre_cp_reference.parquet`** — Row-level frozen benchmark surface for the legal 2024 Pre-CP Reference rolling-validation population. Each row preserves its forecast identity, observed target, selected benchmark identity, selected benchmark prediction, benchmark availability, baseline-tournament support, and selected-benchmark support. Chapter 5 uses this surface for aligned benchmark comparison and fallback.

- **`empirical_forecastability_summary.parquet`** — Summarizes the evidence that motivated carrying the forecasting problems forward, including benchmark MAE, supporting normalized error, skill relative to persistence, the share of Forecast Sequences beating persistence, and short-horizon error stability.

- **`supported_modeling_scope.parquet`** — Defines the approved downstream modeling scope for Sections 4.3–4.5. It carries all **five targets × three horizons × three modeling tracks** into the classical, panel-tree, and neural modeling stages while preserving the appropriate frozen benchmark for each forecasting job.

- **`final_baseline_forecastability_qa.parquet`** — Records the final integrity checks for the complete 4.2.4 package, including benchmark completeness, minimum-MAE winner consistency, shared Post-CP/Full-History benchmark identity, forecastability-summary completeness, downstream scope coverage, and validation of the selected Pre-CP row-level benchmark handoff.

- **`baseline_forecastability_manifest.parquet`** — Compact manifest of the final 4.2.4 handoff assets, including their paths and basic file metadata. This provides a machine-readable inventory for downstream notebooks and reproducibility checks.

#### Intermediate working files

- **`baseline_target_history_lookup.parquet`** — Canonical target-history lookup used to construct the four simple forecasting rules. It preserves the target-aligned historical values needed for daily, weekly, and trailing same-Daypart donors without relying on the broader forecasting feature matrix.

- **`baseline_scoring_support_summary.parquet`** — Compact Track × Stage × Target × Horizon summary of candidate prediction availability. It records individual-rule coverage and the shared-support population used to compare all four candidate baselines on exactly the same observed outcomes.

- **`baseline_predictions__post_cp_primary.parquet`** — Row-level candidate baseline predictions for the Post-CP Primary modeling track. Retained as an audit asset; downstream advanced models should normally use the frozen benchmark registry rather than reopen this full prediction surface.

- **`baseline_predictions__full_history_regime_aware.parquet`** — Row-level candidate baseline predictions for the Full-History Regime-Aware track. This supports audit and comparison work but is not needed to reselect a benchmark, because Full-History deliberately inherits the Post-CP forecasting benchmark.

- **`baseline_predictions__pre_cp_reference.parquet`** — Row-level predictions for all four simple Pre-CP baseline candidates used for benchmark evaluation and detailed baseline diagnostics. Downstream modeling uses `selected_benchmark_predictions__pre_cp_reference.parquet` for the frozen selected benchmark.

**Handoff.** Later forecasting notebooks normally consume the final handoff files. Ordinary forecasting models use `advanced_model_benchmark_registry.parquet` for their benchmark assignment. Chapter 5 additionally uses `selected_benchmark_predictions__pre_cp_reference.parquet` for row-level Pre-CP benchmark comparison and fallback. Intermediate prediction surfaces remain available for detailed baseline diagnostics and reproducibility but are not required downstream production inputs.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=5c43663c-345d-4c21-9332-dd4f928af3ba' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>